# Experimento VQE — versão 20.4 separada

## Núcleo ativo, separabilidade multiplicativa, statevector e QFIM — sem COBYLA

Este notebook separa completamente a investigação causal do pipeline de geração do banco. Ele utiliza somente o arquivo `merge.pkl` já produzido e reconstrói o mesmo Hamiltoniano e o mesmo ansatz do experimento 20.1.

Todas as intervenções são diretas:

$$
\theta \longrightarrow U(\theta) \longrightarrow |\psi(\theta)\rangle,
$$

sem COBYLA, sem reotimização dos demais parâmetros e sem geração de um novo banco.

### Perguntas científicas desta versão

1. Quais parâmetros alteram diretamente a probabilidade, a energia e a distribuição dos portfólios?
2. Os sete parâmetros ativos são suficientes para recuperar a solução quando os outros 23 permanecem aleatórios?
3. Os 23 parâmetros restantes são realmente nulos no **statevector complexo** ou alteram apenas fases invisíveis à medição computacional?
4. Qual é o número de direções independentes do estado segundo a QFIM?
5. Os 21 pares do núcleo ativo são separáveis pelo modelo multiplicativo?
6. A fatorização multiplicativa permanece válida quando 2, 3, ..., 7 parâmetros ativos são modificados simultaneamente?

### Regra de execução

O notebook não importa nem chama qualquer otimizador. Toda avaliação utiliza somente:

```python
assigned = ansatz.assign_parameters(theta, inplace=False)
state = Statevector.from_instruction(assigned)
```

Os resultados já observados nas versões anteriores permanecem descritos em células Markdown, mas as conclusões numéricas também são recalculadas automaticamente. Assim, o texto funciona como guia de leitura e não substitui as tabelas produzidas pela execução atual.


## Como interpretar o circuito antes dos testes

O circuito começa aplicando portas `X` em $k$ qubits. Isso prepara **um estado-base inicial com peso de Hamming $k$**; não significa que a solução ótima já foi inserida no circuito.

Depois, os blocos parametrizados redistribuem a amplitude entre estados que mantêm a cardinalidade:

- `CY`: bloco lógico de **dois qubits**, decomposto com uma rotação controlada `CRY`;
- `CCY`: bloco lógico de **três qubits**, decomposto com `RY` e `CCX`;
- `RY`: operação primitiva de um qubit;
- `CX`: operação primitiva de dois qubits;
- `CCX`: operação primitiva de três qubits.

Portanto, não é correto dizer que toda porta é simplesmente uma junção de dois spins. O Hamiltoniano do portfólio é diagonal e contém termos de um e dois corpos, `Z` e `ZZ`, enquanto o **ansatz** usa blocos de dois e três qubits para navegar no subespaço de Dicke.

O índice $j$ em $\theta_j$ não deve ser fornecido isoladamente ao Transformer como significado físico. O objeto transferível é a descrição estrutural:

$$
(\text{tipo de bloco},\; \text{qubits/ativos},\; \text{distância},\;
\text{posição},\; \text{período},\; \text{termos do Hamiltoniano tocados}).
$$


### Célula 1 — Importações, parâmetros e pastas do experimento

**Em termos simples:** esta célula reúne tudo o que poderá ser alterado antes da execução. Ela não calcula resultados quânticos; apenas carrega bibliotecas, fixa sementes, define os testes e cria as pastas de saída.

**O que é configurado:**

- os parâmetros financeiros $q$, $r_f$ e a cardinalidade $k=4$;
- a fração superior usada para escolher as âncoras;
- quais parâmetros $\theta_j$ serão varridos;
- a resolução das grades;
- a quantidade de repetições dos testes aleatórios;
- os limites usados para declarar separabilidade multiplicativa;
- os diretórios de tabelas, figuras e checkpoints.

A semente `RANDOM_SEED` torna as escolhas aleatórias reproduzíveis. O dicionário `CONFIG` registra a configuração completa que será exportada no final.


In [ ]:
# ============================================================
# 1. IMPORTS E CONFIGURAÇÃO ÚNICA
# ============================================================

from __future__ import annotations

from itertools import combinations
from pathlib import Path
import ast
import hashlib
import json
import math
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import spearmanr

# Identificador usado no manifesto para distinguir esta versão das anteriores.
NOTEBOOK_VERSION = "20.4-isolated"
# Semente única: garante que amostras, ordens e alvos aleatórios possam ser repetidos.
RANDOM_SEED = 42

# Parâmetros do problema financeiro.
# Q_VALUE pondera risco versus retorno; RISK_FREE é o termo constante da função objetivo.
# TARGET_K fixa a seleção em exatamente 4 ativos.
Q_VALUE = 0.5
RISK_FREE = 0.0475
TARGET_K = 4

# Tolerância usada para considerar duas energias numericamente iguais.
ENERGY_ATOL = 1e-8

# ------------------------------------------------------------
# ÚNICO CAMINHO DE ENTRADA
# Altere somente esta linha quando o merge.pkl estiver em outro local.
# ------------------------------------------------------------
MERGE_PKL = Path(r"C:\Users\Marlon_Kelly\Downloads\merge.pkl")

# Seleção dos vetores âncora.
# Primeiro conservamos aproximadamente TOP_FRACTION do banco e depois reavaliamos
# no máximo MAX_EXACT_REEVALUATION candidatos para escolher N_ANCHORS vetores.
TOP_FRACTION = 0.10
N_ANCHORS = 10
MAX_EXACT_REEVALUATION = 100

# Varreduras individuais detalhadas.
# ACTIVE_THETA_INDICES define o núcleo ativo proposto.
# DETAILED_THETA_INDICES inclui ativos e controles que receberão uma grade fina.
ACTIVE_THETA_INDICES = [2, 14, 17, 19, 22, 25, 27]
DETAILED_THETA_INDICES = [17, 2, 14, 19, 22, 25, 27, 3, 24]
DETAILED_STEP = 0.01

# Critério para declarar uma curva individual numericamente plana.
# O limiar relativo é multiplicado pelo maior valor de P(x*) da curva.
SWEEP_FLAT_ABS_TOL = 1e-10
SWEEP_FLAT_REL_TOL = 1e-8
COMMON_PHASE_POINTS = 401

# Atlas estrutural dos 30 parâmetros.
RUN_ALL_THETA_ATLAS = True
ATLAS_GRID_POINTS = 65
ACTIVE_AMPLITUDE_THRESHOLD = 0.50

# Testes acumulados já existentes.
STRONG_ORDER = [17, 2, 14, 19, 22, 25, 27]
CONTROL_ORDER = [24, 0, 1, 9]
GATE_OFF_ORDER = [25, 27]
CUMULATIVE_LAMBDA_POINTS = 101

# Testes do núcleo ativo versus os 23 parâmetros restantes.
RUN_ACTIVE_CORE_TESTS = True
INACTIVE_SINGLE_GRID_POINTS = 33
INACTIVE_NESTED_REPEATS = 20
ACTIVE_CORE_RESCUE_TRIALS = 50

# Comparação do statevector complexo.
# Este teste distingue parâmetros realmente nulos de parâmetros que alteram apenas fases.
RUN_STATEVECTOR_PHASE_TESTS = True
STATEVECTOR_PHASE_TRIALS = 50
STATEVECTOR_SUPPORT_EPSILON = 1e-12

# Matriz de Informação de Fisher Quântica (QFIM).
# A derivada é central e a projeção retira a direção de fase global.
RUN_QFIM_TESTS = True
QFIM_MAX_ANCHORS = 10
QFIM_FINITE_DIFFERENCE_STEP = 1e-6
QFIM_RELATIVE_EIGEN_THRESHOLD = 1e-8
QFIM_ABSOLUTE_EIGEN_THRESHOLD = 1e-10

# Superfícies 2D dos 21 pares do núcleo ativo.
RUN_2D_INTERACTIONS = True
RUN_ALL_ACTIVE_PAIRS = True
INTERACTION_USE_ACTIVE_CORE = True
INTERACTION_PAIRS = (
    list(combinations(ACTIVE_THETA_INDICES, 2))
    if RUN_ALL_ACTIVE_PAIRS
    else [(17, 2), (17, 14)]
)
INTERACTION_PLOT_PAIRS = [(17, 2), (17, 14)]
INTERACTION_GRID_POINTS = 33
INTERACTION_MAX_ANCHORS = 3
SEPARABILITY_EPSILON = 1e-12
LOG_PROBABILITY_THRESHOLD = 1e-10
MULTIPLICATIVE_RELATIVE_ERROR_THRESHOLD = 0.05
SVD_RANK1_EXPLAINED_THRESHOLD = 0.99

# Fatorização simultânea do núcleo ativo.
# Para cada tamanho 2,...,7, sorteia subconjuntos e ângulos e compara a avaliação
# direta com o produto das respostas unidimensionais.
RUN_ACTIVE_FACTOR_MODEL = True
ACTIVE_FACTOR_MAX_ANCHORS = 10
ACTIVE_FACTOR_TRIALS_PER_SIZE = 100
ACTIVE_FACTOR_NUMERIC_FLOOR = 1e-300

# Saídas e checkpoints. A pasta nova impede mistura com checkpoints 20.3.
OUTPUT_ROOT = Path("vqe_r") / "pipeline_v20_4_qfim_factorization"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
DISTRIBUTION_DIR = OUTPUT_ROOT / "mean_distributions"

# Cria toda a árvore de saída antes de iniciar cálculos longos.
for directory in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR, CHECKPOINT_DIR, DISTRIBUTION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Cópia serializável das configurações. Ela será incorporada ao manifesto final.
CONFIG = {
    "notebook_version": NOTEBOOK_VERSION,
    "merge_pkl": str(MERGE_PKL),
    "target_k": TARGET_K,
    "q_value": Q_VALUE,
    "risk_free": RISK_FREE,
    "top_fraction": TOP_FRACTION,
    "n_anchors": N_ANCHORS,
    "active_theta_indices": ACTIVE_THETA_INDICES,
    "detailed_theta_indices": DETAILED_THETA_INDICES,
    "detailed_step": DETAILED_STEP,
    "sweep_flat_abs_tol": SWEEP_FLAT_ABS_TOL,
    "sweep_flat_rel_tol": SWEEP_FLAT_REL_TOL,
    "all_theta_atlas": RUN_ALL_THETA_ATLAS,
    "atlas_grid_points": ATLAS_GRID_POINTS,
    "active_amplitude_threshold": ACTIVE_AMPLITUDE_THRESHOLD,
    "strong_order": STRONG_ORDER,
    "control_order": CONTROL_ORDER,
    "gate_off_order": GATE_OFF_ORDER,
    "inactive_single_grid_points": INACTIVE_SINGLE_GRID_POINTS,
    "inactive_nested_repeats": INACTIVE_NESTED_REPEATS,
    "active_core_rescue_trials": ACTIVE_CORE_RESCUE_TRIALS,
    "run_statevector_phase_tests": RUN_STATEVECTOR_PHASE_TESTS,
    "statevector_phase_trials": STATEVECTOR_PHASE_TRIALS,
    "run_qfim_tests": RUN_QFIM_TESTS,
    "qfim_max_anchors": QFIM_MAX_ANCHORS,
    "qfim_finite_difference_step": QFIM_FINITE_DIFFERENCE_STEP,
    "qfim_relative_eigen_threshold": QFIM_RELATIVE_EIGEN_THRESHOLD,
    "run_all_active_pairs": RUN_ALL_ACTIVE_PAIRS,
    "interaction_use_active_core": INTERACTION_USE_ACTIVE_CORE,
    "interaction_pairs": INTERACTION_PAIRS,
    "interaction_plot_pairs": INTERACTION_PLOT_PAIRS,
    "interaction_grid_points": INTERACTION_GRID_POINTS,
    "interaction_max_anchors": INTERACTION_MAX_ANCHORS,
    "multiplicative_relative_error_threshold": MULTIPLICATIVE_RELATIVE_ERROR_THRESHOLD,
    "svd_rank1_explained_threshold": SVD_RANK1_EXPLAINED_THRESHOLD,
    "run_active_factor_model": RUN_ACTIVE_FACTOR_MODEL,
    "active_factor_max_anchors": ACTIVE_FACTOR_MAX_ANCHORS,
    "active_factor_trials_per_size": ACTIVE_FACTOR_TRIALS_PER_SIZE,
    "optimizer_used": False,
    "cobyla_calls": 0,
}

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print("Saídas:", OUTPUT_ROOT.resolve())


# Parte I — carregar e auditar somente o `merge.pkl`

O carregamento não procura nomes alternativos em vários diretórios. Existe um único caminho configurado em `MERGE_PKL`.

A célula seguinte aceita `DataFrame`, lista de dicionários ou dicionário serializado, mas não reconstrói nem modifica o banco original.


### Célula 2 — Carregamento controlado do banco `merge.pkl`

**Em termos simples:** esta célula abre o único arquivo de entrada e o transforma em um `DataFrame` de trabalho.

Ela verifica se o caminho existe, aceita três formatos serializados (`DataFrame`, lista de registros ou dicionário) e interrompe a execução caso o arquivo esteja vazio ou tenha um tipo inesperado.

**Saída principal:** `merge_df`, que contém o banco original carregado em memória. O arquivo em disco não é modificado.


In [ ]:
# ============================================================
# 2. CARREGAMENTO ÚNICO DO merge.pkl
# ============================================================

# Resolve "~", converte para caminho absoluto e verifica o arquivo antes da leitura.
merge_path = MERGE_PKL.expanduser().resolve()
if not merge_path.is_file():
    raise FileNotFoundError(
        "merge.pkl não encontrado. Caminho configurado: "
        f"{merge_path}"
    )

# O pickle é lido uma única vez. As conversões seguintes ocorrem apenas em memória.
loaded_object = pd.read_pickle(merge_path)

# Padroniza diferentes formatos serializados para um único DataFrame.
if isinstance(loaded_object, pd.DataFrame):
    merge_df = loaded_object.copy()
elif isinstance(loaded_object, list):
    merge_df = pd.DataFrame(loaded_object)
elif isinstance(loaded_object, dict):
    merge_df = pd.DataFrame(loaded_object)
else:
    raise TypeError(
        "O merge.pkl deve conter DataFrame, lista de registros ou dicionário; "
        f"tipo encontrado: {type(loaded_object)}"
    )

if merge_df.empty:
    raise ValueError("O merge.pkl foi carregado, mas está vazio.")

print("Arquivo:", merge_path)
print("Shape:", merge_df.shape)
print("Colunas:", merge_df.columns.tolist())
display(merge_df.head())


### Célula 3 — Identificação das colunas e conversão dos dados

**Em termos simples:** bancos gerados em versões diferentes podem usar nomes diferentes para a mesma informação. Esta célula cria um mapa de aliases e identifica qual coluna representa retorno, covariância, vetor de parâmetros, energia, probabilidade e bitstring.

Também são definidas funções para converter conteúdos salvos como texto em objetos numéricos:

- `parse_tickers`: recupera os nomes dos ativos;
- `parse_vector`: transforma retornos e vetores $\theta$ em arrays;
- `parse_matrix`: reconstrói a matriz de covariância;
- `normalize_bitstring`: padroniza bitstrings para uma sequência de zeros e uns.

**Importante:** essa normalização ocorre apenas na cópia em memória. O `merge.pkl` original permanece intacto.


In [ ]:
# ============================================================
# 3. NORMALIZAÇÃO DO ESQUEMA SEM ALTERAR O ARQUIVO ORIGINAL
# ============================================================

# Cada chave representa um conceito do experimento; a lista contém nomes de
# coluna aceitos para esse mesmo conceito em versões diferentes do banco.
COLUMN_ALIASES = {
    "tickers": ["tickers", "assets", "asset_names"],
    "assets_return": ["assets_return", "assets_returns", "expected_returns", "mu"],
    "covariance": ["covariance", "covariance_matrix", "sigma"],
    "best_parameters": ["best_parameters", "theta", "theta_final"],
    "initial_point": ["initial_point", "initial_theta", "theta_initial"],
    "objective": ["objective_function_value", "energy", "final_energy"],
    "best_objective": ["best_objective_function_value", "exact_energy", "optimal_energy"],
    "p_best": [
        "p_exact_eval",
        "probability_best_answer",
        "probability_best_answer_shots",
        "p_best",
        "prob_best",
    ],
    "gap": ["gap_exact_eval", "energy_gap", "gap"],
    "dominant_bitstring": [
        "most_frequent_bitstring",
        "most_frequen_bitstring",
        "best_answer",
        "dominant_bitstring",
    ],
    "counts": ["counts", "measurement_counts"],
    "status": ["status"],
}


def first_existing_column(frame, aliases, required=False):
    """Retorna o primeiro alias realmente presente no DataFrame."""
    for name in aliases:
        if name in frame.columns:
            return name
    if required:
        raise KeyError(f"Nenhuma das colunas obrigatórias foi encontrada: {aliases}")
    return None


# Resultado final do mapeamento: conceito lógico -> nome real no merge.pkl.
RESOLVED_COLUMNS = {
    key: first_existing_column(
        merge_df,
        aliases,
        required=key in {"tickers", "assets_return", "covariance", "best_parameters"},
    )
    for key, aliases in COLUMN_ALIASES.items()
}


def parse_serialized(value):
    """Converte texto serializado em lista, dicionário ou array quando possível."""
    if isinstance(value, (np.ndarray, list, tuple, dict, pd.Series, pd.Index, pd.DataFrame)):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        for parser in (json.loads, ast.literal_eval):
            try:
                return parser(text)
            except Exception:
                pass
        cleaned = text.strip("[]()")
        arr = np.fromstring(cleaned.replace(",", " "), sep=" ")
        if arr.size:
            return arr
    return value


def parse_tickers(value):
    """Padroniza a lista de tickers e rejeita nomes vazios."""
    parsed = parse_serialized(value)
    if isinstance(parsed, str):
        items = [item.strip() for item in parsed.replace(";", ",").split(",")]
    elif isinstance(parsed, dict):
        items = list(parsed.keys())
    else:
        items = list(parsed)
    tickers = [str(item).strip().strip("'\"") for item in items]
    if not tickers or any(not item for item in tickers):
        raise ValueError(f"Tickers inválidos: {value}")
    return tickers


def parse_vector(value, tickers=None):
    """Converte um vetor salvo em texto, Series, dicionário ou lista para NumPy."""
    parsed = parse_serialized(value)
    if isinstance(parsed, pd.Series):
        if tickers is not None and set(tickers).issubset(set(parsed.index.astype(str))):
            return parsed.reindex(tickers).to_numpy(dtype=float)
        return parsed.to_numpy(dtype=float)
    if isinstance(parsed, dict):
        if tickers is not None and set(tickers).issubset(set(map(str, parsed.keys()))):
            return np.asarray([parsed[ticker] for ticker in tickers], dtype=float)
        return np.asarray(list(parsed.values()), dtype=float)
    return np.asarray(parsed, dtype=float).reshape(-1)


def parse_matrix(value, tickers=None):
    """Reconstrói uma matriz numérica e preserva a ordem dos tickers quando possível."""
    parsed = parse_serialized(value)
    if isinstance(parsed, pd.DataFrame):
        if tickers is not None:
            return parsed.loc[tickers, tickers].to_numpy(dtype=float)
        return parsed.to_numpy(dtype=float)
    if isinstance(parsed, dict):
        frame = pd.DataFrame(parsed)
        if tickers is not None and set(tickers).issubset(frame.index) and set(tickers).issubset(frame.columns):
            return frame.loc[tickers, tickers].to_numpy(dtype=float)
        return frame.to_numpy(dtype=float)
    array = np.asarray(parsed, dtype=float)
    if array.ndim == 1:
        n = int(round(np.sqrt(array.size)))
        if n * n != array.size:
            raise ValueError("Covariância unidimensional não forma uma matriz quadrada.")
        array = array.reshape(n, n)
    return array


def normalize_bitstring(value):
    """Remove prefixos e espaços, retornando apenas bitstrings binários válidos."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value).replace(" ", "").replace("'", "").replace('"', "")
    if text.startswith("0b"):
        text = text[2:]
    return text if set(text).issubset({"0", "1"}) else None


print(json.dumps(RESOLVED_COLUMNS, indent=2, ensure_ascii=False))


### Célula 4 — Reconstrução do problema financeiro e auditoria do banco

**Em termos simples:** a primeira linha válida do banco é usada para recuperar os ativos, o vetor de retornos $\mu$ e a matriz de covariância $\Sigma$.

A covariância é explicitamente simetrizada:

$$
\Sigma_{\mathrm{sim}}=\frac{\Sigma+\Sigma^\mathsf{T}}{2}.
$$

Depois, uma amostra de até 200 linhas recebe uma impressão digital (`hash`). Se aparecer mais de um hash, o banco contém mais de um problema/Hamiltoniano e a execução é interrompida.

A célula também estima a cardinalidade pelos bitstrings salvos e cria `problem_summary_df`, com retorno, variância e conexão de risco de cada ativo.


In [ ]:
# ============================================================
# 4. EXTRAIR O HAMILTONIANO E AUDITAR CONSISTÊNCIA DO BANCO
# ============================================================

# Usa a primeira linha com um vetor theta válido como referência do problema.
first_valid_index = merge_df[
    merge_df[RESOLVED_COLUMNS["best_parameters"]].notna()
].index[0]
first_row = merge_df.loc[first_valid_index]

tickers = parse_tickers(first_row[RESOLVED_COLUMNS["tickers"]])
mu = parse_vector(first_row[RESOLVED_COLUMNS["assets_return"]], tickers=tickers)
sigma = parse_matrix(first_row[RESOLVED_COLUMNS["covariance"]], tickers=tickers)
# Corrige pequenas assimetrias numéricas sem alterar a parte simétrica do risco.
sigma = 0.5 * (sigma + sigma.T)

N_ASSETS = len(tickers)
if mu.shape != (N_ASSETS,):
    raise ValueError(f"Retornos com shape {mu.shape}; esperado {(N_ASSETS,)}.")
if sigma.shape != (N_ASSETS, N_ASSETS):
    raise ValueError(
        f"Covariância com shape {sigma.shape}; esperado {(N_ASSETS, N_ASSETS)}."
    )
if not np.all(np.isfinite(mu)) or not np.all(np.isfinite(sigma)):
    raise ValueError("Retornos ou covariância possuem valores não finitos.")

# A cardinalidade é inferida do bitstring salvo quando possível.
bit_col = RESOLVED_COLUMNS["dominant_bitstring"]
if bit_col is not None:
    saved_bits = merge_df[bit_col].map(normalize_bitstring).dropna()
    inferred_weights = saved_bits.map(lambda value: value.count("1"))
    inferred_k = int(inferred_weights.mode().iloc[0]) if not inferred_weights.empty else TARGET_K
else:
    inferred_k = TARGET_K

if inferred_k != TARGET_K:
    warnings.warn(
        f"A cardinalidade modal inferida foi k={inferred_k}; "
        f"o experimento está configurado para k={TARGET_K}."
    )

# Confirma que uma amostra do merge representa o mesmo problema.
def problem_fingerprint(row):
    """Cria um hash a partir de tickers, retornos e covariância de uma linha."""
    row_tickers = parse_tickers(row[RESOLVED_COLUMNS["tickers"]])
    row_mu = parse_vector(row[RESOLVED_COLUMNS["assets_return"]], tickers=row_tickers)
    row_sigma = parse_matrix(row[RESOLVED_COLUMNS["covariance"]], tickers=row_tickers)
    payload = (
        "|".join(row_tickers).encode("utf-8")
        + np.asarray(row_mu, dtype=np.float64).tobytes()
        + np.asarray(row_sigma, dtype=np.float64).tobytes()
    )
    return hashlib.sha256(payload).hexdigest()[:16]

# Uma amostra aleatória é suficiente para detectar mistura evidente de problemas,
# sem reler e converter necessariamente todas as linhas do banco.
sample_size = min(200, len(merge_df))
sampled_rows = merge_df.sample(sample_size, random_state=RANDOM_SEED)
fingerprints = sampled_rows.apply(problem_fingerprint, axis=1)
if fingerprints.nunique() != 1:
    raise RuntimeError(
        "O merge.pkl contém mais de um Hamiltoniano na amostra auditada. "
        "Este notebook 20.4 executa um problema por vez; filtre o merge antes de continuar."
    )

DATA_HASH = fingerprints.iloc[0]
min_cov_eigenvalue = float(np.linalg.eigvalsh(sigma).min())

# Tabela por ativo usada posteriormente como descrição estrutural do problema.
problem_summary_df = pd.DataFrame({
    "asset_index": np.arange(N_ASSETS, dtype=int),
    "ticker": tickers,
    "return_sum": mu,
    "variance": np.diag(sigma),
    "risk_connection_abs": np.sum(np.abs(sigma), axis=1),
})

print("n =", N_ASSETS, "| k =", TARGET_K)
print("tickers =", tickers)
print("problem_hash =", DATA_HASH)
print("menor autovalor da covariância =", f"{min_cov_eigenvalue:.3e}")
display(problem_summary_df)


# Parte II — referência clássica e rigidez dos pares

## O que esta parte representa

Antes de analisar o circuito quântico, o notebook resolve exatamente o problema clássico. Como existem 10 ativos e o portfólio deve selecionar 4, o número total de soluções válidas é

$$
\binom{10}{4}=210.
$$

Isso significa que é possível avaliar todos os 210 portfólios e conhecer, sem aproximação:

- a energia mínima exata;
- todos os bitstrings ótimos;
- a distância energética entre decisões concorrentes;
- quais pares de ativos possuem decisões mais rígidas.

## Mínimos condicionados de cada par

Para cada par de ativos $(i,j)$, fixamos os valores de decisão $x_i=a$ e $x_j=b$, com $a,b\in\{0,1\}$. Em seguida, procuramos o melhor portfólio que respeita essas duas decisões e continua selecionando exatamente $k$ ativos:

$$
E_{ij}^{ab}
=
\min_{\substack{x_i=a,\;x_j=b\\ \sum_{\ell}x_{\ell}=k}}
E(x).
$$

Assim, cada par possui quatro energias condicionadas:

$$
E_{ij}^{00},\qquad
E_{ij}^{01},\qquad
E_{ij}^{10},\qquad
E_{ij}^{11}.
$$

Esses valores permitem medir quanto custa trocar a decisão de um ativo, dos dois ativos e quão separado está o melhor estado do par em relação ao segundo melhor. Mais adiante, essas métricas serão ligadas aos blocos quânticos que atuam sobre os mesmos qubits.


### Célula 5 — Solução clássica exata e rigidez dos pares de ativos

**Em termos simples:** para 10 ativos escolhendo exatamente 4, todos os portfólios válidos podem ser enumerados:

$$
N_{\mathrm{portfólios}}=\binom{10}{4}=210.
$$

A função objetivo avaliada para cada bitstring é

$$
E(x)=q\,x^\mathsf{T}\Sigma x-(1-q)\,\mu^\mathsf{T}x+r_f,
$$

com a restrição $\sum_i x_i=k$.

Para cada par de ativos $(i,j)$ e cada estado $a,b\in\{0,1\}$, é calculado o melhor portfólio condicionado:

$$
E_{ij}^{ab}
=
\min_{\substack{x_i=a,\;x_j=b\\ \sum_\ell x_\ell=k}}
E(x).
$$

Esses quatro mínimos permitem medir:

- `G_ij`: separação entre o melhor e o segundo melhor estado condicionado do par;
- gaps de trocar apenas $i$, apenas $j$ ou os dois;
- não aditividade da troca conjunta.

**Saídas principais:** `enumeration_df`, `asset_decision_df`, `pair_gap_df`, `exact_energy` e os bitstrings ótimos.


In [ ]:
# ============================================================
# 5. ENUMERAÇÃO CLÁSSICA EXATA E GAPS CONDICIONAIS
# ============================================================

PAIR_STATES = ("00", "01", "10", "11")


def portfolio_objective(x_binary):
    """Calcula E(x)=q*x.T*Sigma*x-(1-q)*mu.T*x+r_f para um portfólio binário."""
    x = np.asarray(x_binary, dtype=float).reshape(-1)
    return float(
        Q_VALUE * x @ sigma @ x
        - (1.0 - Q_VALUE) * mu @ x
        + RISK_FREE
    )


def bitstring_asset_order(x):
    """Converte o vetor binário para a ordem natural dos ativos."""
    return "".join(str(int(value)) for value in np.asarray(x, dtype=int))


def enumerate_portfolios(k_value=TARGET_K):
    """Enumera exatamente todas as combinações de k ativos entre N_ASSETS."""
    rows = []
    for selected_indices in combinations(range(N_ASSETS), int(k_value)):
        x = np.zeros(N_ASSETS, dtype=int)
        x[list(selected_indices)] = 1
        bits = bitstring_asset_order(x)
        rows.append({
            "x_asset_order": tuple(int(v) for v in x),
            "bitstring_asset_order": bits,
            "bitstring_qiskit_order": bits[::-1],
            "selected_assets": tuple(tickers[index] for index in selected_indices),
            "objective": portfolio_objective(x),
        })
    return pd.DataFrame(rows).sort_values(
        ["objective", "bitstring_asset_order"]
    ).reset_index(drop=True)


# Para n=10 e k=4, esta tabela possui C(10,4)=210 linhas.
enumeration_df = enumerate_portfolios(TARGET_K)
exact_energy = float(enumeration_df.iloc[0]["objective"])
optimal_mask = np.isclose(
    enumeration_df["objective"].to_numpy(dtype=float),
    exact_energy,
    atol=ENERGY_ATOL,
    rtol=0.0,
)
optimal_df = enumeration_df.loc[optimal_mask].copy()
exact_asset_bitstrings = sorted(optimal_df["bitstring_asset_order"].unique())
exact_qiskit_bitstrings = sorted(optimal_df["bitstring_qiskit_order"].unique())
exact_x = np.asarray([int(value) for value in exact_asset_bitstrings[0]], dtype=int)
energy_span = float(enumeration_df["objective"].max() - exact_energy)


def build_asset_decision_margins():
    """Mede o custo mínimo de inverter a decisão de cada ativo no ótimo clássico."""
    selected = np.flatnonzero(exact_x == 1)
    excluded = np.flatnonzero(exact_x == 0)
    rows = []
    for asset_index in range(N_ASSETS):
        alternatives = []
        if exact_x[asset_index] == 1:
            for replacement in excluded:
                trial = exact_x.copy()
                trial[asset_index] = 0
                trial[replacement] = 1
                alternatives.append(portfolio_objective(trial))
        else:
            for removed in selected:
                trial = exact_x.copy()
                trial[asset_index] = 1
                trial[removed] = 0
                alternatives.append(portfolio_objective(trial))
        margin = max(min(alternatives) - exact_energy, 0.0)
        rows.append({
            "asset_index": asset_index,
            "ticker": tickers[asset_index],
            "selected_exact": int(exact_x[asset_index]),
            "decision_margin": float(margin),
            "decision_margin_relative": float(margin / max(energy_span, ENERGY_ATOL)),
        })
    return pd.DataFrame(rows).sort_values("decision_margin", ascending=False)


asset_decision_df = build_asset_decision_margins()


def conditional_pair_best(i, j, state):
    """Retorna o melhor portfólio sob x_i=a e x_j=b para um estado ab."""
    a, b = int(state[0]), int(state[1])
    subset = enumeration_df.loc[
        enumeration_df["x_asset_order"].map(
            lambda x: int(x[i]) == a and int(x[j]) == b
        )
    ]
    if subset.empty:
        raise RuntimeError(f"Estado inviável para par {(i, j)}: {state}")
    return subset.iloc[0]


# Para cada par, calculamos E_00, E_01, E_10 e E_11 e derivamos os gaps.
pair_rows = []
for i, j in combinations(range(N_ASSETS), 2):
    states = {state: conditional_pair_best(i, j, state) for state in PAIR_STATES}
    ordered = sorted(PAIR_STATES, key=lambda state: (states[state]["objective"], state))
    best_state, second_state = ordered[:2]
    global_state = f"{exact_x[i]}{exact_x[j]}"
    a_star, b_star = map(int, global_state)
    flip_i = f"{1-a_star}{b_star}"
    flip_j = f"{a_star}{1-b_star}"
    flip_both = f"{1-a_star}{1-b_star}"
    delta_i = max(float(states[flip_i]["objective"] - exact_energy), 0.0)
    delta_j = max(float(states[flip_j]["objective"] - exact_energy), 0.0)
    delta_both = max(float(states[flip_both]["objective"] - exact_energy), 0.0)
    # G_ij mede quão rigidamente o melhor estado condicionado do par se separa do segundo.
    gij = max(float(states[second_state]["objective"] - states[best_state]["objective"]), 0.0)
    pair_rows.append({
        "asset_index_i": i,
        "asset_index_j": j,
        "asset_i": tickers[i],
        "asset_j": tickers[j],
        "pair": f"{tickers[i]}/{tickers[j]}",
        "global_pair_state": global_state,
        "G_ij": gij,
        "G_ij_relative": gij / max(energy_span, ENERGY_ATOL),
        "single_flip_i_gap": delta_i,
        "single_flip_j_gap": delta_j,
        "joint_flip_gap": delta_both,
        "joint_flip_nonadditivity": delta_both - delta_i - delta_j,
        **{f"E_{state}": float(states[state]["objective"]) for state in PAIR_STATES},
    })

pair_gap_df = pd.DataFrame(pair_rows).sort_values("G_ij", ascending=False).reset_index(drop=True)

print("Portfólios válidos:", len(enumeration_df))
print("Energia exata:", exact_energy)
print("Bitstring ótimo — ordem dos ativos:", exact_asset_bitstrings)
print("Bitstring ótimo — ordem Qiskit:", exact_qiskit_bitstrings)
print("Ativos selecionados:", optimal_df.iloc[0]["selected_assets"])
display(asset_decision_df)
display(pair_gap_df.head(15))


# Parte III — reconstrução exata do Hamiltoniano e do ansatz do modelo 20.1

A construção abaixo foi separada do gerador de banco, mas preserva a mesma lógica do notebook 20.1:

- mesmo QUBO/Ising;
- mesmo estado inicial de peso $k$;
- mesmos blocos `CY` e `CCY`;
- mesma ordenação rastreável dos parâmetros;
- mesmo `ANSATZ_SEED`.

O notebook interrompe a execução caso o vetor salvo tenha dimensão incompatível ou caso a auditoria estrutural falhe.


### Célula 6 — Dependências quânticas sem importação de otimizadores

**Em termos simples:** esta célula carrega somente as classes necessárias para montar o QUBO, converter para Ising, construir o circuito e calcular o `Statevector`.

Nenhum método como COBYLA, SPSA ou outro otimizador variacional é importado. Isso garante que as células seguintes apenas atribuam valores de $\theta$ e avaliem diretamente o circuito.


In [ ]:
# ============================================================
# 6. DEPENDÊNCIAS QUÂNTICAS — SEM OTIMIZADOR
# ============================================================

# Dependências para formular o problema binário e construir o circuito.
# Não há importação de classes de otimização variacional.
from docplex.mp.model import Model
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector
from qiskit_optimization.translators import from_docplex_mp
from qiskit_optimization.converters import QuadraticProgramToQubo

try:
    from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver
except ImportError:
    from qiskit.algorithms.minimum_eigensolvers import NumPyMinimumEigensolver

print("Dependências quânticas carregadas. Nenhum otimizador foi importado.")


### Célula 7 — Construção do QUBO e do Hamiltoniano de Ising

**Em termos simples:** o problema financeiro clássico é escrito com variáveis binárias $x_i\in\{0,1\}$ e a restrição de selecionar exatamente $k$ ativos:

$$
\sum_i x_i=k.
$$

O modelo é convertido para QUBO e depois para um Hamiltoniano de Ising:

$$
H=\sum_i h_i Z_i+\sum_{i<j}J_{ij}Z_iZ_j+\text{constante}.
$$

A célula exige que o Hamiltoniano seja diagonal, isto é, sem termos `X` ou `Y`. Em seguida, compara a energia mínima do Ising com a energia obtida pela enumeração dos 210 portfólios. Se elas não coincidirem, a execução para.

**Saídas principais:** `ising`, `ising_offset` e `hamiltonian_terms_df`.


In [ ]:
# ============================================================
# 7. QUBO E HAMILTONIANO DE ISING
# ============================================================


def build_docplex_ising():
    """Constrói o modelo binário, converte para QUBO/Ising e audita a energia exata."""
    model = Model(name=f"portfolio_n{N_ASSETS}_k{TARGET_K}")
    variables = np.array(
        [model.binary_var(name=f"x_{index}") for index in range(N_ASSETS)],
        dtype=object,
    )
    # Termo quadrático de risco x^T Sigma x.
    risk_expression = model.sum(
        float(sigma[row, column]) * variables[row] * variables[column]
        for row in range(N_ASSETS)
        for column in range(N_ASSETS)
    )
    # Termo linear de retorno esperado mu^T x.
    return_expression = model.sum(
        float(mu[index]) * variables[index]
        for index in range(N_ASSETS)
    )
    model.minimize(
        Q_VALUE * risk_expression
        - (1.0 - Q_VALUE) * return_expression
        + RISK_FREE
    )
    # Restrição de cardinalidade: exatamente TARGET_K variáveis devem valer 1.
    model.add_constraint(
        model.sum(variables.tolist()) == int(TARGET_K),
        ctname="budget",
    )

    # Docplex -> QuadraticProgram -> QUBO -> operador Ising e deslocamento constante.
    quadratic_program = from_docplex_mp(model=model)
    qubo = QuadraticProgramToQubo().convert(quadratic_program)
    ising, offset = qubo.to_ising()

    # O problema de portfólio deve gerar apenas termos diagonais I, Z e ZZ.
    labels = [str(label) for label in ising.paulis.to_labels()]
    non_diagonal = [label for label in labels if "X" in label or "Y" in label]
    if non_diagonal:
        raise RuntimeError(f"Hamiltoniano não diagonal: {non_diagonal[:10]}")

    # Auditoria independente: a menor energia Ising deve coincidir com a enumeração.
    exact_result = NumPyMinimumEigensolver().compute_minimum_eigenvalue(operator=ising)
    exact_energy_ising = float(np.real(exact_result.eigenvalue + offset))
    if not np.isclose(exact_energy_ising, exact_energy, atol=1e-10, rtol=0.0):
        raise RuntimeError(
            "Energia Ising e enumeração clássica não coincidem: "
            f"{exact_energy_ising} vs {exact_energy}"
        )

    terms_df = pd.DataFrame({
        "pauli_label": labels,
        "coefficient": np.real(np.asarray(ising.coeffs)).astype(float),
        "body_order": [label.count("Z") for label in labels],
    })
    return model, quadratic_program, qubo, ising, float(offset), terms_df


model, quadratic_program, qubo, ising, ising_offset, hamiltonian_terms_df = build_docplex_ising()
print("Termos Ising:", len(hamiltonian_terms_df))
display(hamiltonian_terms_df.sort_values(["body_order", "pauli_label"]))


### Célula 8 — Construção rastreável do ansatz de Dicke

**Em termos simples:** esta célula reproduz o circuito parametrizado usado no experimento 20.1 e registra a origem estrutural de cada parâmetro.

- `CY_parameterized` cria um bloco lógico de dois qubits cuja operação parametrizada primitiva é `CRY`;
- `CCY_parameterized` cria um bloco lógico de três qubits usando `RY` e `CCX`;
- as portas `X` iniciais preparam apenas um estado-base com peso de Hamming $k$;
- cada parâmetro recebe informações como posição, distância entre qubits e tipo de bloco.

O número esperado de parâmetros é

$$
N_\theta=\frac{k(2n-k-1)}{2}.
$$

**Saídas principais:** `ansatz`, `structure_df`, `initial_x_qubits` e `N_PARAMETERS`.


In [ ]:
# ============================================================
# 8. PORTAS E ANSATZ DE DICKE RASTREÁVEL — CÓPIA DA CABEÇA 20.1
# ============================================================


def CY_parameterized(identifier):
    """Cria o bloco lógico CY com uma rotação controlada CRY(theta)."""
    param = ParameterVector(name=f"x{identifier}", length=1)
    qc = QuantumCircuit(2)
    qc.cry(param[0], 1, 0)
    return qc.to_gate(label="CY")


def CCY_parameterized(identifier):
    """Cria o bloco lógico CCY com rotações RY(theta) e controles CCX."""
    param = ParameterVector(name=f"y{identifier}", length=1)
    qc = QuantumCircuit(3)
    qc.ry(param[0], 0)
    qc.ccx(2, 1, 0)
    qc.ry(-param[0], 0)
    qc.ccx(2, 1, 0)
    return qc.to_gate(label="CCY")


def dicke_parameter_count(n_value, k_value):
    """Número esperado de parâmetros do ansatz: k(2n-k-1)/2."""
    return int(k_value * (2 * n_value - k_value - 1) / 2)


def build_tracked_dicke_ansatz(n_value, k_value, seed):
    """Constrói o ansatz e registra a origem lógica de cada parâmetro."""
    # Preserva o estado aleatório global para que a construção do ansatz não
    # altere outras rotinas aleatórias do notebook.
    numpy_state = np.random.get_state()
    try:
        np.random.seed(int(seed))
        qr = QuantumRegister(n_value, "q")
        qc = QuantumCircuit(qr)

        # Prepara um estado-base com exatamente k excitações; não injeta o ótimo clássico.
        initial_x_qubits = []
        for excitation_index in range(k_value):
            qubit = n_value - excitation_index - 1
            qc.x(qubit)
            initial_x_qubits.append(qubit)

        # Cada bloco criado gera um registro que depois será ligado ao theta_index real.
        records = []
        aux = 1
        for l_value in range(n_value)[::-1]:
            for i_value in range(l_value - 1, l_value - 1 - k_value, -1):
                if i_value >= 0:
                    unique_name = f"{l_value}{i_value}{aux}{np.random.randint(0, int(1e8))}"
                    if i_value == l_value - 1:
                        gate = CY_parameterized(unique_name)
                        gate_parameter = list(gate.params[0].parameters)[0]
                        qc.cx(qr[i_value], qr[l_value])
                        qc.append(gate, [qr[i_value], qr[l_value]])
                        qc.cx(qr[i_value], qr[l_value])
                        gate_type = "CY"
                    else:
                        gate = CCY_parameterized(unique_name)
                        gate_parameter = list(gate.params[0].parameters)[0]
                        qc.cx(qr[i_value], qr[l_value])
                        qc.append(gate, [qr[i_value], qr[i_value + 1], qr[l_value]])
                        qc.cx(qr[i_value], qr[l_value])
                        gate_type = "CCY"
                    records.append({
                        "parameter_object": gate_parameter,
                        "parameter_name": str(gate_parameter),
                        "l": int(l_value),
                        "i": int(i_value),
                        "distance": int(l_value - i_value),
                        "ansatz_gate_type": gate_type,
                    })
                aux += 1
    finally:
        np.random.set_state(numpy_state)

    # A ordem dos parâmetros é extraída do circuito decomposto, a mesma forma usada
    # nas avaliações por Statevector.
    decomposed = qc.decompose()
    ordered_parameters = list(decomposed.parameters)
    parameter_to_index = {
        parameter: index for index, parameter in enumerate(ordered_parameters)
    }
    structure_rows = []
    for record in records:
        record = record.copy()
        parameter_object = record.pop("parameter_object")
        structure_rows.append({
            "theta_index": int(parameter_to_index[parameter_object]),
            **record,
        })

    structure_df = pd.DataFrame(structure_rows).sort_values("theta_index").reset_index(drop=True)
    expected = dicke_parameter_count(n_value, k_value)
    if len(structure_df) != expected:
        raise RuntimeError(f"Esperados {expected} parâmetros; encontrados {len(structure_df)}.")

    structure_df["n"] = int(n_value)
    structure_df["k"] = int(k_value)
    structure_df["rho_k_over_n"] = k_value / n_value
    structure_df["u_l"] = structure_df["l"] / max(n_value - 1, 1)
    structure_df["u_distance"] = structure_df["distance"] / max(k_value, 1)
    structure_df["u_theta_index"] = structure_df["theta_index"] / max(expected - 1, 1)
    return decomposed, structure_df, tuple(sorted(initial_x_qubits))


ANSATZ_SEED = RANDOM_SEED + 100 * N_ASSETS + TARGET_K
ansatz, structure_df, initial_x_qubits = build_tracked_dicke_ansatz(
    N_ASSETS, TARGET_K, ANSATZ_SEED
)
N_PARAMETERS = int(ansatz.num_parameters)

print("ANSATZ_SEED =", ANSATZ_SEED)
print("n_parameters =", N_PARAMETERS)
print("qubits com X inicial =", initial_x_qubits)
print("estado-base inicial em ordem Qiskit =", "".join(
    "1" if qubit in initial_x_qubits else "0"
    for qubit in range(N_ASSETS - 1, -1, -1)
))


### Célula 9 — Mapa físico, lógico e financeiro de cada $\theta_j$

**Em termos simples:** esta célula percorre o circuito decomposto e descobre em qual operação física cada parâmetro aparece, quais qubits ele toca e em que posições do circuito ele é usado.

A periodicidade estrutural é definida aqui:

$$
T_j=
\begin{cases}
4\pi, & \text{se o parâmetro aparece em uma porta CRY},\\
2\pi, & \text{se aparece somente em RY}.
\end{cases}
$$

O período é salvo em `parameter_map_df["angular_period"]`. Mais adiante, ele será usado como limite final da varredura:

```python
grid = inclusive_grid(0.0, period, ...)
```

A célula também liga cada bloco aos pares de ativos contidos em seus qubits e acrescenta métricas clássicas como `max_internal_G_ij`.

**Saídas principais:** `parameter_map_df`, `parameter_occurrence_df` e `structural_audit`.


In [ ]:
# ============================================================
# 9. MAPA FÍSICO, LÓGICO E FINANCEIRO DOS PARÂMETROS
# ============================================================


def build_physical_parameter_map(ansatz, structure_df):
    """Liga cada theta a operações primitivas, qubits, período e posição no circuito."""
    parameter_order = list(ansatz.parameters)
    parameter_to_index = {
        parameter: index for index, parameter in enumerate(parameter_order)
    }
    occurrence_rows = []

    # Percorre cada instrução do circuito decomposto e registra todas as ocorrências
    # de parâmetros nas expressões angulares das portas.
    for instruction_position, instruction in enumerate(ansatz.data):
        operation = instruction.operation
        operation_name = str(operation.name).lower()
        qubits = tuple(
            int(ansatz.find_bit(qubit).index)
            for qubit in instruction.qubits
        )
        for parameter_slot, expression in enumerate(operation.params):
            for parameter in getattr(expression, "parameters", set()):
                if parameter not in parameter_to_index:
                    continue
                try:
                    coefficient = float(expression.gradient(parameter))
                except Exception:
                    coefficient = np.nan
                occurrence_rows.append({
                    "theta_index": int(parameter_to_index[parameter]),
                    "parameter_name": str(parameter),
                    "primitive_operation": operation_name,
                    "primitive_qubits": qubits,
                    "instruction_position": int(instruction_position),
                    "parameter_slot": int(parameter_slot),
                    "parameter_coefficient": coefficient,
                })

    # Uma linha por ocorrência física de parâmetro.
    occurrence_df = pd.DataFrame(occurrence_rows)
    physical_rows = []
    for theta_index, group in occurrence_df.groupby("theta_index"):
        operations = sorted(set(group["primitive_operation"].astype(str)))
        primitive_qubits = tuple(sorted({
            int(qubit)
            for qubit_tuple in group["primitive_qubits"]
            for qubit in qubit_tuple
        }))
        # RY(theta+2pi) difere apenas por fase global, mas CRY pode exigir 4pi porque
        # o sinal relativo entre os setores do qubit de controle é observável.
        if "cry" in operations:
            primitive_type = "CRY"
            angular_period = float(4 * np.pi)
            periodicity_class = "four_pi_eligible"
        else:
            primitive_type = "RY"
            angular_period = float(2 * np.pi)
            periodicity_class = "guaranteed_2pi"
        physical_rows.append({
            "theta_index": int(theta_index),
            "primitive_physical_type": primitive_type,
            "primitive_operations": tuple(operations),
            "primitive_parameter_qubits": primitive_qubits,
            "angular_period": angular_period,
            "periodicity_structural_class": periodicity_class,
            "n_occurrences_decomposed": int(len(group)),
            "first_instruction": int(group["instruction_position"].min()),
            "last_instruction": int(group["instruction_position"].max()),
        })

    # Une a descrição lógica do bloco à descrição física observada após decomposição.
    parameter_map = structure_df.merge(
        pd.DataFrame(physical_rows),
        on="theta_index",
        how="left",
        validate="one_to_one",
    ).sort_values("theta_index").reset_index(drop=True)

    def logical_qubits(row):
        """Retorna todos os qubits pertencentes ao bloco lógico CY ou CCY."""
        i_value, l_value = int(row["i"]), int(row["l"])
        if row["ansatz_gate_type"] == "CY":
            return tuple(sorted((i_value, l_value)))
        return tuple(sorted((i_value, i_value + 1, l_value)))

    parameter_map["logical_block_qubits"] = parameter_map.apply(logical_qubits, axis=1)
    parameter_map["logical_assets"] = parameter_map["logical_block_qubits"].map(
        lambda qubits: tuple(tickers[index] for index in qubits)
    )
    parameter_map["block_size"] = parameter_map["logical_block_qubits"].map(len)

    # Auditoria estrutural: qualquer inconsistência interrompe o experimento.
    checks = {
        "all_parameters_mapped": bool(len(parameter_map) == N_PARAMETERS),
        "CY_is_CRY": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CY"),
            "primitive_physical_type",
        ].eq("CRY").all()),
        "CCY_is_RY": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CCY"),
            "primitive_physical_type",
        ].eq("RY").all()),
        "CY_has_two_logical_qubits": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CY"),
            "block_size",
        ].eq(2).all()),
        "CCY_has_three_logical_qubits": bool(parameter_map.loc[
            parameter_map["ansatz_gate_type"].eq("CCY"),
            "block_size",
        ].eq(3).all()),
    }
    failed = [name for name, value in checks.items() if not value]
    if failed:
        raise RuntimeError(f"Auditoria estrutural falhou: {failed}")

    return parameter_map, occurrence_df, checks


parameter_map_df, parameter_occurrence_df, structural_audit = build_physical_parameter_map(
    ansatz, structure_df
)

# Liga cada bloco aos pares clássicos contidos nos seus qubits lógicos.
def pair_metrics_for_block(qubits):
    """Resume os gaps clássicos dos pares de ativos internos ao bloco."""
    pairs = {tuple(sorted(pair)) for pair in combinations(qubits, 2)}
    subset = pair_gap_df.loc[
        pair_gap_df.apply(
            lambda row: tuple(sorted((int(row["asset_index_i"]), int(row["asset_index_j"])))) in pairs,
            axis=1,
        )
    ]
    return pd.Series({
        "n_internal_asset_pairs": int(len(subset)),
        "max_internal_G_ij": float(subset["G_ij"].max()),
        "mean_internal_G_ij": float(subset["G_ij"].mean()),
        "max_internal_joint_gap": float(subset["joint_flip_gap"].max()),
        "max_internal_nonadditivity_abs": float(subset["joint_flip_nonadditivity"].abs().max()),
        "internal_pairs": tuple(subset["pair"].tolist()),
    })

block_pair_metrics = parameter_map_df["logical_block_qubits"].apply(pair_metrics_for_block)
parameter_map_df = pd.concat([parameter_map_df, block_pair_metrics], axis=1)

print(json.dumps(structural_audit, indent=2, ensure_ascii=False))
display(parameter_map_df)


## Auditoria visual da criação do circuito

A tabela anterior é a ponte entre o índice local e a descrição transferível:

- `theta_index`: posição local neste circuito;
- `ansatz_gate_type`: bloco lógico `CY` ou `CCY`;
- `primitive_physical_type`: operação parametrizada observada após decomposição;
- `logical_block_qubits`: qubits realmente envolvidos pelo bloco;
- `logical_assets`: ativos associados a esses qubits;
- `max_internal_G_ij`: maior rigidez clássica entre os pares internos do bloco;
- `angular_period`: domínio máximo usado na varredura.

A célula seguinte mostra o circuito e contabiliza as operações. O estado preparado pelas portas `X` é apenas a semente de peso $k$, não o bitstring ótimo clássico.


### Célula 10 — Auditoria visual do circuito

**Em termos simples:** esta célula mostra quantas operações de cada tipo existem, exibe a tabela estrutural dos parâmetros e tenta desenhar o ansatz.

A tabela permite conferir, para cada `theta_index`, o bloco lógico, a operação física, os qubits/ativos tocados, o período angular e a rigidez clássica interna.

Caso o desenho com Matplotlib não esteja disponível, o circuito é mostrado em formato de texto.


In [ ]:
# ============================================================
# 10. VISUALIZAÇÃO E CONTAGEM DAS PORTAS
# ============================================================

# Contagem de portas no circuito já decomposto.
operation_counts = pd.DataFrame(
    sorted(ansatz.count_ops().items()),
    columns=["operation", "count"],
)

display(operation_counts)
display(parameter_map_df[[
    "theta_index",
    "ansatz_gate_type",
    "primitive_physical_type",
    "logical_block_qubits",
    "logical_assets",
    "distance",
    "first_instruction",
    "last_instruction",
    "angular_period",
    "internal_pairs",
    "max_internal_G_ij",
]])

# Tenta o desenho gráfico; o modo texto é um fallback para ambientes sem suporte.
try:
    display(ansatz.draw(output="mpl", fold=120))
except Exception as exc:
    warnings.warn(f"Desenho matplotlib não disponível: {type(exc).__name__}: {exc}")
    print(ansatz.draw(output="text", fold=120))


# Parte IV — compatibilidade entre `merge.pkl` e o circuito reconstruído

Antes de interpretar qualquer varredura, o notebook verifica:

1. todos os vetores possuem a dimensão esperada;
2. vetores salvos podem ser atribuídos ao circuito;
3. a distribuição exata recalculada é compatível com as colunas salvas;
4. nenhuma avaliação chama otimizador.


### Célula 11 — Avaliação exata de um vetor $\theta$

**Em termos simples:** esta é a função central usada em todas as intervenções posteriores. Ela recebe um vetor $\theta$, atribui os valores ao ansatz e calcula diretamente o estado quântico:

$$
|\psi(\theta)\rangle=U(\theta)|\psi_0\rangle.
$$

A função `evaluate_theta` devolve, entre outras métricas:

- probabilidade total dos bitstrings ótimos;
- energia esperada $\langle H\rangle$ e gap para a energia exata;
- bitstring dominante;
- massa dentro do subespaço válido de cardinalidade $k$;
- entropia e razão de participação;
- massa acumulada nos 1, 5 e 10 melhores portfólios clássicos.

Quando existe uma distribuição de referência, também são calculadas a distância de variação total

$$
\operatorname{TVD}(p,q)=\frac{1}{2}\sum_z|p_z-q_z|
$$

e a divergência de Jensen–Shannon.

Ao final, uma pequena amostra do banco é reavaliada para verificar compatibilidade entre os vetores salvos e o circuito reconstruído.


In [ ]:
# ============================================================
# 11. FUNÇÕES DE PARSE DOS VETORES E MÉTRICAS EXATAS
# ============================================================

# Converte todos os vetores salvos e mantém apenas aqueles compatíveis com o
# número de parâmetros do ansatz reconstruído.
best_parameters_column = RESOLVED_COLUMNS["best_parameters"]
merge_work_df = merge_df.copy()
merge_work_df["theta_vector"] = merge_work_df[best_parameters_column].map(parse_vector)
merge_work_df["theta_dimension"] = merge_work_df["theta_vector"].map(len)

dimension_counts = merge_work_df["theta_dimension"].value_counts().sort_index()
print("Dimensões encontradas:")
display(dimension_counts.rename_axis("theta_dimension").to_frame("rows"))

merge_work_df = merge_work_df.loc[
    merge_work_df["theta_dimension"].eq(N_PARAMETERS)
].copy()
if merge_work_df.empty:
    raise RuntimeError(
        f"Nenhum vetor do merge possui a dimensão esperada de {N_PARAMETERS} parâmetros."
    )

# Mapeamento entre índices do Statevector e bitstrings na convenção do Qiskit.
all_labels = np.asarray(
    [format(index, f"0{N_ASSETS}b") for index in range(2 ** N_ASSETS)],
    dtype=object,
)
label_to_index = {str(label): int(index) for index, label in enumerate(all_labels)}
valid_bitstrings = enumeration_df["bitstring_qiskit_order"].astype(str).to_numpy(dtype=object)
valid_indices = np.asarray([label_to_index[bitstring] for bitstring in valid_bitstrings], dtype=int)
valid_objectives = enumeration_df["objective"].to_numpy(dtype=float)
optimal_indices = np.asarray([label_to_index[bitstring] for bitstring in exact_qiskit_bitstrings], dtype=int)
optimal_set = set(exact_qiskit_bitstrings)


def total_variation_distance(p, q):
    """Calcula TVD(p,q)=0.5*sum(|p-q|), entre 0 e 1."""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return float(0.5 * np.sum(np.abs(p - q)))


def jensen_shannon_divergence(p, q, epsilon=1e-15):
    """Calcula uma divergência simétrica e finita entre duas distribuições."""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(p.sum(), epsilon)
    q = q / max(q.sum(), epsilon)
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + epsilon) / (m + epsilon)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + epsilon) / (m + epsilon)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))


def circular_distance(a, b, period):
    """Menor distância entre dois ângulos em um círculo de período conhecido."""
    return float(abs((float(a) - float(b) + 0.5 * period) % period - 0.5 * period))


def shortest_delta_to_target(source, target, period):
    """Deslocamento assinado mais curto da origem até o alvo periódico."""
    return float((float(target) - float(source) + 0.5 * period) % period - 0.5 * period)


def statevector_from_theta(theta):
    """Retorna o vetor de estado complexo após atribuição direta dos parâmetros."""
    theta = np.asarray(theta, dtype=float).reshape(-1)
    if len(theta) != N_PARAMETERS:
        raise ValueError(f"Theta com dimensão {len(theta)}; esperado {N_PARAMETERS}.")
    assigned = ansatz.assign_parameters(theta, inplace=False)
    return np.asarray(Statevector.from_instruction(assigned).data, dtype=np.complex128)


def evaluate_theta(
    theta,
    reference_probability=None,
    return_valid_probability=False,
    return_statevector=False,
):
    """Atribui theta ao ansatz e calcula exatamente estado, energia e probabilidades."""
    theta = np.asarray(theta, dtype=float).reshape(-1)
    if len(theta) != N_PARAMETERS:
        raise ValueError(f"Theta com dimensão {len(theta)}; esperado {N_PARAMETERS}.")

    # Intervenção direta: não existe passo de otimização entre atribuir theta e avaliar.
    state_data = statevector_from_theta(theta)
    state = Statevector(state_data)
    probability = np.asarray(state.probabilities(), dtype=float)
    # Restringe a distribuição aos C(n,k) bitstrings que respeitam a cardinalidade.
    valid_probability = probability[valid_indices]
    valid_mass = float(valid_probability.sum())
    leakage = max(0.0, 1.0 - valid_mass)
    p_optimal = float(probability[optimal_indices].sum())
    # Energia física completa = valor esperado do operador + offset da conversão QUBO.
    energy = float(np.real(state.expectation_value(ising)) + ising_offset)
    dominant_index = int(np.argmax(probability))
    dominant_bitstring = str(all_labels[dominant_index])

    nonzero = valid_probability[valid_probability > 0.0]
    entropy = float(-np.sum(nonzero * np.log(nonzero)))
    normalized_entropy = float(entropy / np.log(len(valid_probability)))
    participation = float(1.0 / np.sum(valid_probability ** 2))

    dominant_valid_position = int(np.argmax(valid_probability))
    dominant_valid_rank = dominant_valid_position + 1

    result = {
        "p_optimal": p_optimal,
        "expected_energy": energy,
        "energy_gap": float(abs(energy - exact_energy)),
        "dominant_bitstring": dominant_bitstring,
        "dominant_probability": float(probability[dominant_index]),
        "dominant_is_optimal": bool(dominant_bitstring in optimal_set),
        "dominant_valid_rank": dominant_valid_rank,
        "p_top_1_classical": float(valid_probability[:1].sum()),
        "p_top_5_classical": float(valid_probability[:5].sum()),
        "p_top_10_classical": float(valid_probability[:10].sum()),
        "valid_dicke_mass": valid_mass,
        "leakage_outside_k": leakage,
        "normalized_entropy_valid": normalized_entropy,
        "participation_ratio_valid": participation,
    }
    if reference_probability is not None:
        result["tvd_vs_anchor"] = total_variation_distance(probability, reference_probability)
        result["jsd_vs_anchor"] = jensen_shannon_divergence(probability, reference_probability)
    if return_valid_probability:
        result["valid_probability"] = valid_probability.astype(np.float32)
    if return_statevector:
        result["statevector"] = state_data
    result["full_probability"] = probability
    return result


# Auditoria em uma amostra pequena antes das campanhas longas.
# Compara probabilidade e bitstring salvos com a reavaliação exata atual.
audit_indices = merge_work_df.sample(min(10, len(merge_work_df)), random_state=RANDOM_SEED).index
compatibility_rows = []
for row_index in audit_indices:
    row = merge_work_df.loc[row_index]
    metrics = evaluate_theta(row["theta_vector"])
    saved_p = (
        pd.to_numeric(pd.Series([row[RESOLVED_COLUMNS["p_best"]]]), errors="coerce").iloc[0]
        if RESOLVED_COLUMNS["p_best"] is not None
        else np.nan
    )
    saved_bit = (
        normalize_bitstring(row[RESOLVED_COLUMNS["dominant_bitstring"]])
        if RESOLVED_COLUMNS["dominant_bitstring"] is not None
        else None
    )
    compatibility_rows.append({
        "row_index": row_index,
        "saved_p": saved_p,
        "exact_p_recomputed": metrics["p_optimal"],
        "p_difference": metrics["p_optimal"] - saved_p if np.isfinite(saved_p) else np.nan,
        "saved_dominant": saved_bit,
        "exact_dominant_recomputed": metrics["dominant_bitstring"],
        "dominant_equal": bool(saved_bit == metrics["dominant_bitstring"]) if saved_bit else np.nan,
        "exact_energy_recomputed": metrics["expected_energy"],
        "leakage": metrics["leakage_outside_k"],
    })

compatibility_audit_df = pd.DataFrame(compatibility_rows)
display(compatibility_audit_df)

if compatibility_audit_df["leakage"].max() > 1e-10:
    raise RuntimeError("O circuito reconstruído apresentou vazamento para fora do subespaço k=4.")


### Célula 12 — Máscara dos 10% melhores e seleção das âncoras

**Em termos simples:** esta célula reduz o banco aos vetores mais promissores antes das varreduras caras.

Para cada linha válida são construídos dois rankings percentuais:

- $R_p$: cresce quando a probabilidade salva da solução ótima aumenta;
- $R_{\mathrm{gap}}$: cresce quando o gap de energia diminui.

O score salvo é

$$
Q_{\mathrm{salvo}}=\frac{R_p+R_{\mathrm{gap}}}{2}.
$$

A máscara mantém aproximadamente a fração superior configurada:

$$
Q_{\mathrm{salvo}}\geq
\operatorname{quantil}_{1-\texttt{TOP\_FRACTION}}(Q_{\mathrm{salvo}}).
$$

Os melhores candidatos são reavaliados exatamente com `evaluate_theta`. Um novo score é calculado com as métricas exatas e os `N_ANCHORS` melhores vetores tornam-se as âncoras.

**Importante:** a máscara seleciona linhas do banco; ela não zera componentes internos do vetor $\theta$.


In [ ]:
# ============================================================
# 12. MÁSCARA DOS 10% MELHORES E SELEÇÃO DE 10 ÂNCORAS
# ============================================================

# Recupera os nomes reais das colunas que alimentarão a seleção.
objective_col = RESOLVED_COLUMNS["objective"]
best_objective_col = RESOLVED_COLUMNS["best_objective"]
p_col = RESOLVED_COLUMNS["p_best"]
gap_col = RESOLVED_COLUMNS["gap"]
status_col = RESOLVED_COLUMNS["status"]

bank = merge_work_df.copy()
bank["saved_p"] = (
    pd.to_numeric(bank[p_col], errors="coerce") if p_col is not None else np.nan
)
bank["saved_objective"] = (
    pd.to_numeric(bank[objective_col], errors="coerce") if objective_col is not None else np.nan
)

# Prioridade para o gap:
# 1) usa a coluna pronta; 2) calcula objetivo - melhor objetivo;
# 3) compara a energia salva com a referência clássica exata.
if gap_col is not None:
    bank["saved_gap"] = pd.to_numeric(bank[gap_col], errors="coerce").abs()
elif best_objective_col is not None and objective_col is not None:
    bank["saved_gap"] = (
        pd.to_numeric(bank[objective_col], errors="coerce")
        - pd.to_numeric(bank[best_objective_col], errors="coerce")
    ).abs()
else:
    bank["saved_gap"] = (bank["saved_objective"] - exact_energy).abs()

if status_col is not None:
    status_mask = bank[status_col].astype(str).str.lower().eq("ok")
else:
    status_mask = pd.Series(True, index=bank.index)

# Primeira máscara booleana: status válido, theta presente e gap calculável.
valid_bank = bank.loc[
    status_mask
    & bank["theta_vector"].notna()
    & bank["saved_gap"].notna()
].copy()

if valid_bank.empty:
    raise RuntimeError("Nenhum vetor válido foi encontrado para a seleção das âncoras.")

# Ranking percentual da probabilidade: valores maiores recebem ranks maiores.
if valid_bank["saved_p"].notna().any():
    valid_bank["p_quality_rank"] = valid_bank["saved_p"].rank(
        pct=True, ascending=True, method="average"
    )
else:
    valid_bank["p_quality_rank"] = 0.5

# Ranking invertido do gap: gaps menores recebem ranks maiores.
valid_bank["gap_quality_rank"] = valid_bank["saved_gap"].rank(
    pct=True, ascending=False, method="average"
)
# Score de pré-seleção com pesos iguais para probabilidade e proximidade energética.
valid_bank["quality_score_saved"] = 0.5 * (
    valid_bank["p_quality_rank"] + valid_bank["gap_quality_rank"]
)

# Limiar do quantil superior. Com TOP_FRACTION=0.10, usa o percentil 90.
threshold = float(valid_bank["quality_score_saved"].quantile(1.0 - TOP_FRACTION))
top_masked_bank = valid_bank.loc[
    valid_bank["quality_score_saved"].ge(threshold)
].copy()

# Limita a reavaliação exata para controlar o custo computacional.
candidate_pool = top_masked_bank.sort_values(
    ["quality_score_saved", "saved_p", "saved_gap"],
    ascending=[False, False, True],
).head(min(MAX_EXACT_REEVALUATION, len(top_masked_bank)))

# Reavalia todos os candidatos sob o mesmo circuito e a mesma referência.
exact_candidate_rows = []
for row_index, row in candidate_pool.iterrows():
    metrics = evaluate_theta(row["theta_vector"])
    exact_candidate_rows.append({
        "source_row_index": row_index,
        "theta_vector": row["theta_vector"],
        "saved_p": row["saved_p"],
        "saved_gap": row["saved_gap"],
        "quality_score_saved": row["quality_score_saved"],
        **{key: value for key, value in metrics.items() if key != "full_probability"},
    })

exact_candidates_df = pd.DataFrame(exact_candidate_rows)
exact_candidates_df["p_rank_exact"] = exact_candidates_df["p_optimal"].rank(
    pct=True, ascending=True
)
exact_candidates_df["gap_rank_exact"] = exact_candidates_df["energy_gap"].rank(
    pct=True, ascending=False
)
exact_candidates_df["quality_score_exact"] = 0.5 * (
    exact_candidates_df["p_rank_exact"] + exact_candidates_df["gap_rank_exact"]
)

# Seleção definitiva: score exato, probabilidade maior e gap menor.
anchors_df = exact_candidates_df.sort_values(
    ["quality_score_exact", "p_optimal", "energy_gap"],
    ascending=[False, False, True],
).head(min(N_ANCHORS, len(exact_candidates_df))).reset_index(drop=True)
anchors_df.insert(0, "anchor_id", np.arange(len(anchors_df), dtype=int))

if len(anchors_df) < N_ANCHORS:
    warnings.warn(f"Somente {len(anchors_df)} âncoras válidas foram encontradas.")

print("linhas válidas no banco:", len(valid_bank))
print("linhas após máscara superior:", len(top_masked_bank))
print("candidatos reavaliados exatamente:", len(exact_candidates_df))
print("âncoras finais:", len(anchors_df))
display(anchors_df.drop(columns=["theta_vector"], errors="ignore"))


# Parte V — varreduras individuais detalhadas

Para cada uma das 10 âncoras, somente um componente é alterado. Os outros 29 permanecem exatamente fixos.

Parâmetros detalhados:

\[
\{17,2,14,19,22,25,27,3,24\}.
\]

Cada parâmetro é varrido sobre o período estrutural atribuído ao seu bloco. Os resultados são salvos por tarefa `(anchor_id, theta_index)`, permitindo retomar a execução.

A normalização min–max só é aplicada quando a amplitude da curva supera o limiar numérico. Curvas planas são mantidas em zero, evitando amplificar ruído de ponto flutuante.

As curvas de períodos \(2\pi\) e \(4\pi\) são interpoladas para uma grade comum de fase entre 0 e 1 antes da média entre âncoras.


### Célula 13 — Varredura individual de um parâmetro por vez

**Em termos simples:** para cada âncora e cada parâmetro selecionado, somente $\theta_j$ é alterado. Todos os outros componentes permanecem fixos.

Para uma âncora $\theta^{(a)}$, a célula avalia

$$
\theta^{(a,j)}(\phi)
=
\bigl(
\theta_0^{(a)},\ldots,\theta_{j-1}^{(a)},
\phi,
\theta_{j+1}^{(a)},\ldots
\bigr),
\qquad 0\leq\phi\leq T_j.
$$

O limite $T_j$ vem de `parameter_map_df["angular_period"]`: `RY` usa $2\pi$ e `CRY` pode usar $4\pi$.

Cada tarefa salva:

- um `.pkl` com métricas escalares;
- um `.npz` com a distribuição dos 210 portfólios válidos em cada ponto da grade.

Os checkpoints evitam repetir tarefas já concluídas. Nenhum otimizador é chamado.


In [ ]:
# ============================================================
# 13. MOTOR DE VARREDURA INDIVIDUAL — SEM COBYLA
# ============================================================


def inclusive_grid(start, end, step=None, n_points=None):
    """Cria uma grade que inclui explicitamente os limites start e end."""
    start, end = float(start), float(end)
    if step is not None:
        values = np.arange(start, end, float(step), dtype=float)
        if len(values) == 0 or not np.isclose(values[-1], end, atol=1e-12, rtol=0.0):
            values = np.append(values, end)
        else:
            values[-1] = end
        return values
    if n_points is None or n_points < 2:
        raise ValueError("Informe step ou n_points >= 2.")
    return np.linspace(start, end, int(n_points), endpoint=True)


def parameter_period(theta_index):
    """Recupera em parameter_map_df o período 2pi ou 4pi do theta informado."""
    row = parameter_map_df.loc[
        parameter_map_df["theta_index"].eq(int(theta_index))
    ]
    if len(row) != 1:
        raise KeyError(f"theta_{theta_index} não foi identificado de forma única.")
    return float(row.iloc[0]["angular_period"])


def run_single_parameter_task(anchor_row, theta_index, step=DETAILED_STEP):
    """Varre um único theta de uma âncora, mantendo os demais parâmetros fixos."""
    anchor_id = int(anchor_row["anchor_id"])
    theta_index = int(theta_index)
    # O limite final não é fixado diretamente aqui: ele vem do mapa físico.
    # RY -> 2*pi; qualquer ocorrência CRY -> 4*pi.
    period = parameter_period(theta_index)
    grid = inclusive_grid(0.0, period, step=step)

    task_stem = f"anchor_{anchor_id:02d}_theta_{theta_index:02d}"
    summary_path = CHECKPOINT_DIR / f"detailed_{task_stem}.pkl"
    distribution_path = CHECKPOINT_DIR / f"detailed_{task_stem}_valid_probabilities.npz"

    # Checkpoint por combinação âncora × parâmetro.
    if summary_path.exists() and distribution_path.exists():
        return pd.read_pickle(summary_path), distribution_path

    # Distribuição da âncora usada como referência para TVD e JSD.
    theta_anchor = np.asarray(anchor_row["theta_vector"], dtype=float).copy()
    anchor_metrics = evaluate_theta(theta_anchor)
    reference_probability = anchor_metrics["full_probability"]

    rows = []
    probability_matrix = np.empty((len(grid), len(valid_indices)), dtype=np.float32)
    # One-at-a-time: apenas theta_test[theta_index] muda em cada avaliação.
    for grid_index, value in enumerate(grid):
        theta_test = theta_anchor.copy()
        theta_test[theta_index] = float(value)
        metrics = evaluate_theta(
            theta_test,
            reference_probability=reference_probability,
            return_valid_probability=True,
        )
        # A distribuição grande fica numa matriz separada, não dentro do DataFrame.
        probability_matrix[grid_index] = metrics.pop("valid_probability")
        metrics.pop("full_probability")
        rows.append({
            "anchor_id": anchor_id,
            "source_row_index": anchor_row["source_row_index"],
            "theta_index": theta_index,
            "grid_index": int(grid_index),
            "theta_value": float(value),
            "theta_over_period": float(value / period),
            "theta_over_pi": float(value / np.pi),
            "period": period,
            "anchor_theta_raw": float(theta_anchor[theta_index]),
            "anchor_theta_canonical": float(theta_anchor[theta_index] % period),
            "distance_to_anchor": circular_distance(value, theta_anchor[theta_index], period),
            **metrics,
            "optimizer_used": False,
        })

    task_df = pd.DataFrame(rows)
    # Normalização min-max local. Curvas numericamente planas não devem ser
    # amplificadas, pois dividir ruído de máquina por um span ~1e-15 cria serrilhado.
    p_min = float(task_df["p_optimal"].min())
    p_max = float(task_df["p_optimal"].max())
    span = p_max - p_min
    flat_threshold = max(
        SWEEP_FLAT_ABS_TOL,
        SWEEP_FLAT_REL_TOL * max(abs(p_max), 1.0),
    )
    is_flat = bool(span <= flat_threshold)
    task_df["p_span"] = span
    task_df["flat_threshold"] = flat_threshold
    task_df["is_flat_sweep"] = is_flat
    if not is_flat:
        task_df["p_optimal_normalized"] = np.clip(
            (task_df["p_optimal"] - p_min) / span,
            0.0,
            1.0,
        )
    else:
        # Zero significa ausência de sensibilidade normalizada.
        task_df["p_optimal_normalized"] = 0.0

    task_df.to_pickle(summary_path)
    np.savez_compressed(
        distribution_path,
        valid_probability=probability_matrix,
        theta_grid=grid,
        valid_bitstrings=valid_bitstrings,
        valid_objectives=valid_objectives,
    )
    return task_df, distribution_path


for theta_index in DETAILED_THETA_INDICES:
    if not 0 <= theta_index < N_PARAMETERS:
        raise IndexError(f"theta_{theta_index} não existe no circuito com {N_PARAMETERS} parâmetros.")

all_detailed_frames = []
detailed_distribution_paths = {}
for _, anchor_row in anchors_df.iterrows():
    for theta_index in DETAILED_THETA_INDICES:
        task_df, distribution_path = run_single_parameter_task(anchor_row, theta_index)
        all_detailed_frames.append(task_df)
        detailed_distribution_paths[(int(anchor_row["anchor_id"]), int(theta_index))] = distribution_path
        print(
            f"concluído: âncora {int(anchor_row['anchor_id'])} | "
            f"theta_{theta_index} | {len(task_df)} pontos"
        )

individual_sweep_df = pd.concat(all_detailed_frames, ignore_index=True)
print("Total de avaliações detalhadas:", len(individual_sweep_df))


### Célula 14 — Resumo dos picos e comparação entre âncoras

**Em termos simples:** esta célula transforma cada curva individual em números fáceis de comparar.

Para cada combinação `âncora × parâmetro`, ela identifica o máximo, o mínimo, o valor em zero, a amplitude, a posição do pico, a mudança de energia e a maior distância para a distribuição da âncora.

Como os ângulos são periódicos, o centro dos picos é calculado de forma circular, evitando tratar fases próximas de 0 e 1 como distantes.

Também são produzidos:

- uma curva normalizada média para cada parâmetro;
- um gráfico bruto com as 10 âncoras e a curva média.


In [ ]:
# ============================================================
# 14. SISTEMATICIDADE, PICOS, ZERO E CURVAS NORMALIZADAS
# ============================================================

# Resume cada curva âncora × theta em um único registro.
individual_anchor_rows = []
for (anchor_id, theta_index), group in individual_sweep_df.groupby(["anchor_id", "theta_index"]):
    group = group.sort_values("grid_index")
    period = float(group["period"].iloc[0])
    peak_row = group.loc[group["p_optimal"].idxmax()]
    minimum_row = group.loc[group["p_optimal"].idxmin()]
    zero_row = group.iloc[(group["theta_value"] - 0.0).abs().argmin()]
    anchor_theta = float(group["anchor_theta_canonical"].iloc[0])
    individual_anchor_rows.append({
        "anchor_id": int(anchor_id),
        "theta_index": int(theta_index),
        "period": period,
        "p_min": float(minimum_row["p_optimal"]),
        "p_max": float(peak_row["p_optimal"]),
        "p_amplitude": float(peak_row["p_optimal"] - minimum_row["p_optimal"]),
        "p_at_zero": float(zero_row["p_optimal"]),
        "zero_fraction_of_peak": float(zero_row["p_optimal"] / max(peak_row["p_optimal"], 1e-15)),
        "peak_angle": float(peak_row["theta_value"]),
        "peak_phase": float(peak_row["theta_over_period"] % 1.0),
        "minimum_angle": float(minimum_row["theta_value"]),
        "anchor_theta": anchor_theta,
        "anchor_to_peak_distance": circular_distance(anchor_theta, peak_row["theta_value"], period),
        "energy_range": float(group["expected_energy"].max() - group["expected_energy"].min()),
        "tvd_max": float(group["tvd_vs_anchor"].max()),
        "entropy_range": float(
            group["normalized_entropy_valid"].max()
            - group["normalized_entropy_valid"].min()
        ),
    })

individual_anchor_summary_df = pd.DataFrame(individual_anchor_rows)

# Mediana circular aproximada pela fase com menor soma de distâncias.
def circular_medoid(phases):
    """Escolhe a fase observada com menor soma de distâncias circulares."""
    phases = np.asarray(phases, dtype=float) % 1.0
    distances = np.abs(phases[:, None] - phases[None, :])
    distances = np.minimum(distances, 1.0 - distances)
    return float(phases[np.argmin(distances.sum(axis=1))])

# Agrega as 10 âncoras para medir sistematicidade de cada parâmetro.
systematic_rows = []
for theta_index, group in individual_anchor_summary_df.groupby("theta_index"):
    peak_center = circular_medoid(group["peak_phase"].to_numpy())
    peak_distances = np.minimum(
        np.abs(group["peak_phase"].to_numpy() - peak_center),
        1.0 - np.abs(group["peak_phase"].to_numpy() - peak_center),
    )
    systematic_rows.append({
        "theta_index": int(theta_index),
        "n_anchors": int(len(group)),
        "median_p_amplitude": float(group["p_amplitude"].median()),
        "median_energy_range": float(group["energy_range"].median()),
        "median_tvd_max": float(group["tvd_max"].median()),
        "median_zero_fraction_of_peak": float(group["zero_fraction_of_peak"].median()),
        "fraction_zero_below_5pct_peak": float((group["zero_fraction_of_peak"] <= 0.05).mean()),
        "peak_phase_medoid": peak_center,
        "peak_phase_median_distance": float(np.median(peak_distances)),
        "fraction_peaks_within_5pct_period": float((peak_distances <= 0.05).mean()),
        "median_anchor_to_peak_distance": float(group["anchor_to_peak_distance"].median()),
    })

individual_systematic_summary_df = pd.DataFrame(systematic_rows).merge(
    parameter_map_df,
    on="theta_index",
    how="left",
    validate="one_to_one",
).sort_values("median_p_amplitude", ascending=False)

display(individual_anchor_summary_df)
display(individual_systematic_summary_df[[
    "theta_index",
    "ansatz_gate_type",
    "primitive_physical_type",
    "logical_assets",
    "median_p_amplitude",
    "median_zero_fraction_of_peak",
    "fraction_zero_below_5pct_peak",
    "peak_phase_medoid",
    "peak_phase_median_distance",
    "fraction_peaks_within_5pct_period",
    "median_anchor_to_peak_distance",
]])

# Curvas médias em uma grade comum de fase.
# O uso de uma grade comum evita comparar parâmetros de período 2π e 4π por
# índices de grade diferentes. Curvas planas permanecem em zero.
common_phase = np.linspace(0.0, 1.0, COMMON_PHASE_POINTS)
mean_shape_rows = []
for theta_index, theta_group in individual_sweep_df.groupby("theta_index"):
    interpolated_curves = []
    flat_flags = []
    amplitudes = []
    for anchor_id, anchor_group in theta_group.groupby("anchor_id"):
        anchor_group = anchor_group.sort_values("theta_over_period")
        phase = anchor_group["theta_over_period"].to_numpy(dtype=float)
        normalized = anchor_group["p_optimal_normalized"].to_numpy(dtype=float)
        # A fase final 1 representa o mesmo ponto periódico da fase 0, mas é
        # mantida para fechar visualmente a curva.
        interpolated = np.interp(common_phase, phase, normalized)
        interpolated_curves.append(np.clip(interpolated, 0.0, 1.0))
        flat_flags.append(bool(anchor_group["is_flat_sweep"].iloc[0]))
        amplitudes.append(float(anchor_group["p_span"].iloc[0]))

    mean_curve = np.mean(np.vstack(interpolated_curves), axis=0)
    mean_shape_rows.append({
        "theta_index": int(theta_index),
        "mean_curve": mean_curve,
        "median_amplitude": float(np.median(amplitudes)),
        "fraction_flat_anchors": float(np.mean(flat_flags)),
    })

mean_shape_df = pd.DataFrame(mean_shape_rows).sort_values(
    "median_amplitude", ascending=False
)

responsive_indices = mean_shape_df.loc[
    mean_shape_df["fraction_flat_anchors"].lt(0.5), "theta_index"
].astype(int).tolist()
flat_indices = mean_shape_df.loc[
    mean_shape_df["fraction_flat_anchors"].ge(0.5), "theta_index"
].astype(int).tolist()

fig, (ax_shape, ax_amplitude) = plt.subplots(
    2,
    1,
    figsize=(12, 9),
    gridspec_kw={"height_ratios": [3.2, 1.2]},
)

# Painel superior: somente formas com amplitude acima do ruído numérico.
for _, row in mean_shape_df.iterrows():
    theta_index = int(row["theta_index"])
    if theta_index in responsive_indices:
        ax_shape.plot(
            common_phase,
            row["mean_curve"],
            linewidth=2.0,
            label=f"theta_{theta_index}",
        )

# Uma única linha de referência representa todos os parâmetros planos.
if flat_indices:
    ax_shape.plot(
        common_phase,
        np.zeros_like(common_phase),
        linestyle="--",
        linewidth=1.8,
        label="planos: " + ", ".join(f"theta_{i}" for i in flat_indices),
    )

ax_shape.set_xlabel("fase angular normalizada: theta / período estrutural")
ax_shape.set_ylabel("resposta min–max por âncora")
ax_shape.set_title("Forma média das varreduras individuais em 10 vetores")
ax_shape.set_ylim(-0.02, 1.02)
ax_shape.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))

# Painel inferior: amplitude absoluta, que impede interpretar ruído como efeito.
ordered = mean_shape_df.sort_values("theta_index")
amplitude_floor = max(SWEEP_FLAT_ABS_TOL * 0.1, np.finfo(float).tiny)
ax_amplitude.bar(
    ordered["theta_index"].astype(str),
    np.maximum(ordered["median_amplitude"], amplitude_floor),
)
ax_amplitude.axhline(
    SWEEP_FLAT_ABS_TOL,
    linestyle="--",
    linewidth=1.2,
    label="limiar de curva plana",
)
ax_amplitude.set_yscale("log")
ax_amplitude.set_xlabel("parâmetro")
ax_amplitude.set_ylabel("amplitude mediana de P(x*)")
ax_amplitude.legend(loc="best")

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "individual_sweeps_mean_shape_and_absolute_amplitude.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

print("Parâmetros responsivos no conjunto detalhado:", responsive_indices)
print("Parâmetros numericamente planos no conjunto detalhado:", flat_indices)

# Curvas brutas: preservam diferenças absolutas de desempenho entre as âncoras.
for theta_index, group in individual_sweep_df.groupby("theta_index"):
    fig, ax = plt.subplots(figsize=(10, 5))
    for anchor_id, anchor_group in group.groupby("anchor_id"):
        anchor_group = anchor_group.sort_values("theta_over_period")
        ax.plot(
            anchor_group["theta_over_period"],
            np.clip(anchor_group["p_optimal"], 0.0, 1.0),
            alpha=0.25,
        )
    mean_curve = group.groupby("grid_index", as_index=False).agg(
        phase=("theta_over_period", "first"),
        p_mean=("p_optimal", "mean"),
    ).sort_values("phase")
    ax.plot(
        mean_curve["phase"],
        np.clip(mean_curve["p_mean"], 0.0, 1.0),
        linewidth=2.5,
        label="média das âncoras",
    )
    ax.set_xlabel("theta / período estrutural")
    ax.set_ylabel("P(x*)")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(f"Varredura direta de theta_{theta_index} — 10 vetores, sem COBYLA")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"theta_{theta_index:02d}_ten_anchors_raw.png", dpi=180)
    plt.show()


#### Leitura do gráfico corrigido

O painel superior compara somente a forma das curvas realmente responsivas. Parâmetros planos são representados por uma única linha em zero, em vez de terem o ruído numérico amplificado pela normalização.

O painel inferior mostra a amplitude absoluta em escala logarítmica. Assim, uma curva só deve ser interpretada como sensível quando também possui amplitude acima do limiar indicado.

Nos resultados já observados, \(\theta_3\) e \(\theta_{24}\) foram planos, enquanto os sete parâmetros do núcleo apresentaram resposta periódica de grande amplitude.


### Célula 15 — Distribuição média sobre os 210 portfólios

**Em termos simples:** para cada parâmetro detalhado, esta célula abre as distribuições salvas das 10 âncoras e calcula a média ponto a ponto.

A matriz final tem a forma

$$
\overline P_j(\phi,z)
=
\frac{1}{N_{\mathrm{âncoras}}}
\sum_a P(z\mid\theta^{(a,j)}(\phi)).
$$

As linhas/colunas ligam cada ângulo aos 210 portfólios ordenados pela energia clássica. O mapa de calor usa $\log_{10}(P+10^{-12})$ para tornar visíveis probabilidades muito pequenas.


In [ ]:
# ============================================================
# 15. DENSIDADE MÉDIA DOS 210 PORTFÓLIOS POR THETA
# ============================================================

# Para cada theta, empilha as matrizes das âncoras e calcula a média.
mean_distribution_metadata = []
for theta_index in DETAILED_THETA_INDICES:
    matrices = []
    theta_grid = None
    for anchor_id in anchors_df["anchor_id"].astype(int):
        path = detailed_distribution_paths[(anchor_id, theta_index)]
        data = np.load(path, allow_pickle=True)
        matrices.append(np.asarray(data["valid_probability"], dtype=np.float64))
        if theta_grid is None:
            theta_grid = np.asarray(data["theta_grid"], dtype=float)
    # Eixo 0 = âncoras; o resultado mantém grade angular × portfólio válido.
    mean_matrix = np.mean(np.stack(matrices, axis=0), axis=0)
    mean_path = DISTRIBUTION_DIR / f"theta_{theta_index:02d}_mean_valid_distribution.npz"
    np.savez_compressed(
        mean_path,
        mean_probability=mean_matrix.astype(np.float32),
        theta_grid=theta_grid,
        valid_bitstrings=valid_bitstrings,
        valid_objectives=valid_objectives,
    )
    mean_distribution_metadata.append({
        "theta_index": theta_index,
        "n_anchors": len(matrices),
        "n_grid": len(theta_grid),
        "n_valid_portfolios": mean_matrix.shape[1],
        "path": str(mean_path.resolve()),
    })

    fig, ax = plt.subplots(figsize=(11, 6))
    # Escala logarítmica para revelar simultaneamente probabilidades grandes e pequenas.
    image = ax.imshow(
        np.log10(mean_matrix.T + 1e-12),
        aspect="auto",
        origin="upper",
        extent=[theta_grid[0], theta_grid[-1], len(valid_bitstrings), 1],
    )
    ax.set_xlabel(f"theta_{theta_index} em radianos")
    ax.set_ylabel("ranking clássico de energia")
    ax.set_title(f"Log10 da probabilidade média — theta_{theta_index}, 10 âncoras")
    fig.colorbar(image, ax=ax, label="log10(probabilidade + 1e-12)")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"theta_{theta_index:02d}_mean_probability_heatmap.png", dpi=180)
    plt.show()

mean_distribution_metadata_df = pd.DataFrame(mean_distribution_metadata)
display(mean_distribution_metadata_df)


# Parte VI — atlas estrutural dos 30 parâmetros

A varredura detalhada responde às hipóteses específicas. Para investigar **por que** alguns parâmetros importam, todos os parâmetros são também varridos em uma grade comum de 65 pontos por período.

O objetivo não é substituir os testes causais anteriores, mas gerar um atlas comparável entre:

- amplitude de $P(x^*)$;
- variação de energia;
- TVD máxima em relação à âncora;
- tipo de bloco;
- qubits e ativos tocados;
- distância do bloco;
- posição no circuito;
- rigidez clássica dos pares internos.


### Célula 16 — Atlas comum dos 30 parâmetros

**Em termos simples:** a varredura detalhada usa alta resolução em alguns parâmetros. Esta célula aplica uma grade comum de `ATLAS_GRID_POINTS` a todos os 30 parâmetros e a todas as âncoras.

Para cada $\theta_j$, são resumidas:

- amplitude de $P(x^*)$;
- faixa da energia;
- mudança máxima da distribuição;
- faixa de entropia;
- posição do pico;
- erro ao comparar duas metades do período.

O atlas é usado para identificar quais parâmetros são empiricamente ativos. A célula compara o conjunto detectado pelo limiar com `ACTIVE_THETA_INDICES` e emite um aviso se houver divergência.


In [ ]:
# ============================================================
# 16. ATLAS DOS 30 THETAS EM GRADE COMUM
# ============================================================


def run_atlas_task(anchor_row, theta_index):
    """Executa a varredura comum de um theta para uma âncora e salva checkpoint."""
    anchor_id = int(anchor_row["anchor_id"])
    theta_index = int(theta_index)
    period = parameter_period(theta_index)
    # Todos os parâmetros usam a mesma quantidade de pontos, mesmo com períodos distintos.
    grid = inclusive_grid(0.0, period, n_points=ATLAS_GRID_POINTS)
    path = CHECKPOINT_DIR / f"atlas_anchor_{anchor_id:02d}_theta_{theta_index:02d}.pkl"
    if path.exists():
        return pd.read_pickle(path)

    theta_anchor = np.asarray(anchor_row["theta_vector"], dtype=float).copy()
    anchor_metrics = evaluate_theta(theta_anchor)
    reference_probability = anchor_metrics["full_probability"]
    rows = []
    for grid_index, value in enumerate(grid):
        theta_test = theta_anchor.copy()
        theta_test[theta_index] = float(value)
        metrics = evaluate_theta(theta_test, reference_probability=reference_probability)
        metrics.pop("full_probability")
        rows.append({
            "anchor_id": anchor_id,
            "theta_index": theta_index,
            "grid_index": grid_index,
            "theta_value": float(value),
            "phase": float(value / period),
            "period": period,
            **metrics,
        })
    frame = pd.DataFrame(rows)
    frame.to_pickle(path)
    return frame


# O atlas pode ser desligado pela configuração para evitar uma campanha longa.
atlas_sweep_df = pd.DataFrame()
if RUN_ALL_THETA_ATLAS:
    atlas_frames = []
    for _, anchor_row in anchors_df.iterrows():
        for theta_index in range(N_PARAMETERS):
            atlas_frames.append(run_atlas_task(anchor_row, theta_index))
        print(f"atlas concluído para âncora {int(anchor_row['anchor_id'])}")
    atlas_sweep_df = pd.concat(atlas_frames, ignore_index=True)

    atlas_anchor_metrics = []
    for (anchor_id, theta_index), group in atlas_sweep_df.groupby(["anchor_id", "theta_index"]):
        group = group.sort_values("grid_index")
        peak = group.loc[group["p_optimal"].idxmax()]
        minimum = group.loc[group["p_optimal"].idxmin()]
        # Compara a primeira e a segunda metade para verificar se a resposta repete
        # após T/2, mesmo quando a estrutura permite um período total maior.
        half = (len(group) - 1) // 2
        first_half = group["p_optimal"].to_numpy()[:half]
        second_half = group["p_optimal"].to_numpy()[half:2 * half]
        half_period_rmse = float(np.sqrt(np.mean((first_half - second_half) ** 2))) if half else np.nan
        atlas_anchor_metrics.append({
            "anchor_id": int(anchor_id),
            "theta_index": int(theta_index),
            "p_amplitude": float(peak["p_optimal"] - minimum["p_optimal"]),
            "energy_range": float(group["expected_energy"].max() - group["expected_energy"].min()),
            "tvd_max": float(group["tvd_vs_anchor"].max()),
            "entropy_range": float(group["normalized_entropy_valid"].max() - group["normalized_entropy_valid"].min()),
            "peak_phase": float(peak["phase"] % 1.0),
            "half_period_rmse": half_period_rmse,
        })

    atlas_anchor_metrics_df = pd.DataFrame(atlas_anchor_metrics)
    atlas_parameter_summary_df = atlas_anchor_metrics_df.groupby("theta_index", as_index=False).agg(
        p_amplitude_median=("p_amplitude", "median"),
        p_amplitude_mean=("p_amplitude", "mean"),
        energy_range_median=("energy_range", "median"),
        tvd_max_median=("tvd_max", "median"),
        entropy_range_median=("entropy_range", "median"),
        half_period_rmse_median=("half_period_rmse", "median"),
        peak_phase_median=("peak_phase", "median"),
    ).merge(
        parameter_map_df,
        on="theta_index",
        how="left",
        validate="one_to_one",
    ).sort_values("p_amplitude_median", ascending=False)

    display(atlas_parameter_summary_df[[
        "theta_index",
        "ansatz_gate_type",
        "primitive_physical_type",
        "logical_assets",
        "distance",
        "p_amplitude_median",
        "energy_range_median",
        "tvd_max_median",
        "half_period_rmse_median",
        "max_internal_G_ij",
        "max_internal_joint_gap",
    ]])


# Auditoria explícita: os sete parâmetros configurados devem coincidir com
# aqueles que ultrapassam o limiar de amplitude no atlas.
# aqueles cuja amplitude mediana ultrapassa o limiar empírico.
if RUN_ALL_THETA_ATLAS and not atlas_parameter_summary_df.empty:
    detected_active = sorted(
        atlas_parameter_summary_df.loc[
            atlas_parameter_summary_df["p_amplitude_median"].ge(ACTIVE_AMPLITUDE_THRESHOLD),
            "theta_index",
        ].astype(int).tolist()
    )
    configured_active = sorted(map(int, ACTIVE_THETA_INDICES))
    active_set_audit_df = pd.DataFrame({
        "configured_active": pd.Series(configured_active, dtype="Int64"),
        "detected_active": pd.Series(detected_active, dtype="Int64"),
    })
    print("Ativos configurados:", configured_active)
    print("Ativos detectados pelo atlas:", detected_active)
    if configured_active != detected_active:
        warnings.warn(
            "O conjunto ativo configurado não coincide com o conjunto detectado. "
            "Revise ACTIVE_THETA_INDICES ou ACTIVE_AMPLITUDE_THRESHOLD antes de interpretar os testes."
        )
else:
    active_set_audit_df = pd.DataFrame()


#### Leitura do atlas já observado

O atlas anterior mostrou uma separação quase binária:

\[
\{2,14,17,19,22,25,27\}
\]

com amplitude próxima de 1, enquanto os demais parâmetros ficaram no piso numérico.

Essa classificação é empírica e local à instância atual. A QFIM será usada para verificar se a mesma separação aparece na geometria do statevector.


### Célula 17 — Relação entre estrutura da porta e relevância empírica

**Em termos simples:** esta célula procura explicar por que alguns parâmetros têm maior efeito que outros.

Ela calcula correlações de Spearman entre respostas do atlas e características estruturais, como distância do bloco, posição no circuito, período, número de ocorrências e rigidez clássica dos pares internos.

Também agrupa os parâmetros por tipo de bloco (`CY`/`CCY`) e operação física (`CRY`/`RY`).

**Cuidado:** uma correlação alta neste conjunto pequeno descreve associação; não prova causalidade isoladamente.


In [ ]:
# ============================================================
# 17. RELEVÂNCIA DAS PORTAS: CORRELAÇÕES E AGRUPAMENTOS
# ============================================================

# As análises abaixo só existem quando o atlas foi realmente calculado.
if RUN_ALL_THETA_ATLAS and not atlas_sweep_df.empty:
    numerical_predictors = [
        "distance",
        "u_l",
        "u_distance",
        "u_theta_index",
        "block_size",
        "n_occurrences_decomposed",
        "first_instruction",
        "last_instruction",
        "angular_period",
        "max_internal_G_ij",
        "mean_internal_G_ij",
        "max_internal_joint_gap",
        "max_internal_nonadditivity_abs",
    ]
    responses = [
        "p_amplitude_median",
        "energy_range_median",
        "tvd_max_median",
        "entropy_range_median",
    ]
    # Spearman mede associação monotônica e não exige relação linear.
    correlation_rows = []
    for response in responses:
        for predictor in numerical_predictors:
            valid = atlas_parameter_summary_df[[response, predictor]].dropna()
            if len(valid) < 3 or valid[predictor].nunique() < 2:
                rho, p_value = np.nan, np.nan
            else:
                rho, p_value = spearmanr(valid[predictor], valid[response])
            correlation_rows.append({
                "response": response,
                "predictor": predictor,
                "n": int(len(valid)),
                "spearman_rho": float(rho) if np.isfinite(rho) else np.nan,
                "p_value": float(p_value) if np.isfinite(p_value) else np.nan,
            })
    gate_relevance_correlations_df = pd.DataFrame(correlation_rows).sort_values(
        "spearman_rho", key=lambda series: series.abs(), ascending=False
    )

    # Comparação agregada entre famílias estruturais de portas.
    gate_type_summary_df = atlas_parameter_summary_df.groupby(
        ["ansatz_gate_type", "primitive_physical_type"], as_index=False
    ).agg(
        n_parameters=("theta_index", "count"),
        p_amplitude_median=("p_amplitude_median", "median"),
        energy_range_median=("energy_range_median", "median"),
        tvd_max_median=("tvd_max_median", "median"),
        half_period_rmse_median=("half_period_rmse_median", "median"),
        internal_G_median=("max_internal_G_ij", "median"),
    )

    display(gate_relevance_correlations_df.head(30))
    display(gate_type_summary_df)

    fig, ax = plt.subplots(figsize=(11, 5))
    ordered = atlas_parameter_summary_df.sort_values("p_amplitude_median", ascending=False)
    ax.bar(ordered["theta_index"].astype(str), ordered["p_amplitude_median"])
    ax.set_xlabel("theta_index")
    ax.set_ylabel("amplitude mediana de P(x*)")
    ax.set_title("Atlas estrutural: sensibilidade individual dos 30 parâmetros")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "atlas_all_theta_p_amplitude.png", dpi=180)
    plt.show()
else:
    gate_relevance_correlations_df = pd.DataFrame()
    gate_type_summary_df = pd.DataFrame()


# Parte VII — suficiência do núcleo ativo e robustez aos 23 parâmetros restantes

O conjunto ativo será fixado, por âncora, nos máximos encontrados no atlas:

$$
\mathcal{A}=\{2,14,17,19,22,25,27\}.
$$

Os outros 23 parâmetros formarão o conjunto inativo $\mathcal{I}$. Serão realizados três testes complementares:

1. **Varredura condicional individual:** cada parâmetro inativo será varrido sozinho enquanto os sete ativos permanecem nos seus picos.
2. **Perturbação aleatória aninhada:** serão modificados progressivamente 1, 2, 3, ..., 23 parâmetros inativos, em várias ordens e alvos aleatórios reproduzíveis.
3. **Teste de resgate do núcleo ativo:** todos os 30 parâmetros serão inicialmente aleatorizados; em seguida somente os sete ativos serão recolocados nos seus melhores ângulos, mantendo os outros 23 aleatórios.

O terceiro teste é o mais direto para avaliar suficiência: ele verifica se o núcleo ativo consegue recuperar a solução sem depender dos valores originais dos demais parâmetros.

### Célula 18 — Construção do núcleo ativo de sete parâmetros

**Em termos simples:** para cada âncora, os sete parâmetros ativos são colocados nos ângulos que maximizaram $P(x^*)$ no atlas. Os outros 23 permanecem nos valores originais da âncora.

O vetor resultante é o núcleo ativo condicionado àquela âncora. A célula compara:

- probabilidade da âncora original;
- probabilidade após ajustar simultaneamente os sete ativos;
- gap de energia;
- mudança da distribuição;
- posição do bitstring dominante no ranking clássico.

Os vetores são guardados em `active_core_theta_by_anchor` para os testes seguintes.


In [ ]:
# ============================================================
# 18. CONSTRUÇÃO DO NÚCLEO ATIVO POR ÂNCORA
# ============================================================

if RUN_ACTIVE_CORE_TESTS and (not RUN_ALL_THETA_ATLAS or atlas_sweep_df.empty):
    raise RuntimeError("Os testes do núcleo ativo exigem RUN_ALL_THETA_ATLAS=True.")

# Partição completa dos 30 parâmetros em 7 ativos e 23 inativos.
ACTIVE_SET = tuple(sorted(map(int, ACTIVE_THETA_INDICES)))
INACTIVE_SET = tuple(index for index in range(N_PARAMETERS) if index not in ACTIVE_SET)

if len(ACTIVE_SET) != 7 or len(INACTIVE_SET) != 23:
    raise RuntimeError(
        f"Esperados 7 ativos e 23 inativos; encontrados {len(ACTIVE_SET)} e {len(INACTIVE_SET)}."
    )


def atlas_extreme_angle(anchor_id, theta_index, kind="best"):
    """Busca no atlas o ângulo de maior ou menor P(x*) para uma âncora."""
    group = atlas_sweep_df.loc[
        atlas_sweep_df["anchor_id"].eq(int(anchor_id))
        & atlas_sweep_df["theta_index"].eq(int(theta_index))
    ]
    if group.empty:
        raise KeyError(f"Atlas ausente para âncora {anchor_id}, theta_{theta_index}.")
    if kind == "best":
        row = group.loc[group["p_optimal"].idxmax()]
    elif kind == "worst":
        row = group.loc[group["p_optimal"].idxmin()]
    else:
        raise ValueError(kind)
    return float(row["theta_value"])


def build_active_core_theta(anchor_row):
    """Substitui somente os sete ativos pelos melhores ângulos do atlas."""
    theta = np.asarray(anchor_row["theta_vector"], dtype=float).copy()
    anchor_id = int(anchor_row["anchor_id"])
    for theta_index in ACTIVE_SET:
        theta[theta_index] = atlas_extreme_angle(anchor_id, theta_index, kind="best")
    return theta


# Constrói um núcleo específico para cada âncora e compara com o vetor original.
active_core_rows = []
active_core_theta_by_anchor = {}
for _, anchor_row in anchors_df.iterrows():
    anchor_id = int(anchor_row["anchor_id"])
    theta_anchor = np.asarray(anchor_row["theta_vector"], dtype=float)
    anchor_metrics = evaluate_theta(theta_anchor)
    theta_core = build_active_core_theta(anchor_row)
    core_metrics = evaluate_theta(
        theta_core,
        reference_probability=anchor_metrics["full_probability"],
    )
    active_core_theta_by_anchor[anchor_id] = theta_core
    active_core_rows.append({
        "anchor_id": anchor_id,
        "active_indices": ACTIVE_SET,
        "inactive_indices": INACTIVE_SET,
        "p_anchor": float(anchor_metrics["p_optimal"]),
        "p_active_core": float(core_metrics["p_optimal"]),
        "active_core_over_anchor": float(
            core_metrics["p_optimal"] / max(anchor_metrics["p_optimal"], 1e-15)
        ),
        "energy_gap_anchor": float(anchor_metrics["energy_gap"]),
        "energy_gap_active_core": float(core_metrics["energy_gap"]),
        "tvd_core_vs_anchor": float(core_metrics["tvd_vs_anchor"]),
        "dominant_rank_active_core": int(core_metrics["dominant_valid_rank"]),
        "p_top_10_active_core": float(core_metrics["p_top_10_classical"]),
    })

active_core_baseline_df = pd.DataFrame(active_core_rows)
display(active_core_baseline_df)
print("Ativos:", ACTIVE_SET)
print("Inativos:", INACTIVE_SET)


### Célula 19 — Os 23 parâmetros continuam inativos com o núcleo fixado?

**Em termos simples:** esta célula mantém os sete parâmetros ativos em seus melhores ângulos e varre um parâmetro inativo por vez sobre todo o seu período.

Isso testa uma hipótese condicional: um parâmetro que parecia inativo perto da âncora original pode tornar-se relevante depois que o núcleo ativo muda.

Para cada inativo são medidos amplitude de $P(x^*)$, pior queda relativa ao núcleo, faixa de energia, TVD máxima, pior ranking dominante e menor massa nos 10 melhores portfólios.


In [ ]:
# ============================================================
# 19. VARREDURA CONDICIONAL DOS 23 PARÂMETROS INATIVOS
# ============================================================


def run_inactive_conditional_sweep(anchor_row, theta_index):
    """Varre um inativo enquanto os sete ativos permanecem fixos no núcleo."""
    anchor_id = int(anchor_row["anchor_id"])
    theta_index = int(theta_index)
    path = CHECKPOINT_DIR / (
        f"v204_inactive_conditional_anchor_{anchor_id:02d}_theta_{theta_index:02d}.pkl"
    )
    if path.exists():
        return pd.read_pickle(path)

    # Ponto de referência deste teste: núcleo ativo, não a âncora original.
    theta_core = active_core_theta_by_anchor[anchor_id].copy()
    core_metrics = evaluate_theta(theta_core)
    reference_probability = core_metrics["full_probability"]
    period = parameter_period(theta_index)
    grid = inclusive_grid(0.0, period, n_points=INACTIVE_SINGLE_GRID_POINTS)

    rows = []
    for grid_index, value in enumerate(grid):
        theta_test = theta_core.copy()
        theta_test[theta_index] = float(value)
        metrics = evaluate_theta(theta_test, reference_probability=reference_probability)
        metrics.pop("full_probability")
        rows.append({
            "anchor_id": anchor_id,
            "theta_index": theta_index,
            "grid_index": int(grid_index),
            "theta_value": float(value),
            "phase": float(value / period),
            "period": float(period),
            "active_core_fixed": True,
            **metrics,
        })

    frame = pd.DataFrame(rows)
    # Razão 1 significa desempenho igual ao núcleo; valores menores indicam degradação.
    frame["p_relative_to_core"] = frame["p_optimal"] / max(float(core_metrics["p_optimal"]), 1e-15)
    frame.to_pickle(path)
    return frame


inactive_conditional_sweep_df = pd.DataFrame()
inactive_conditional_summary_df = pd.DataFrame()
if RUN_ACTIVE_CORE_TESTS:
    frames = []
    for _, anchor_row in anchors_df.iterrows():
        for theta_index in INACTIVE_SET:
            frames.append(run_inactive_conditional_sweep(anchor_row, theta_index))
        print(f"varreduras condicionais concluídas para âncora {int(anchor_row['anchor_id'])}")
    inactive_conditional_sweep_df = pd.concat(frames, ignore_index=True)

    # Primeiro resume cada âncora; depois usa a mediana entre âncoras.
    anchor_summaries = []
    for (anchor_id, theta_index), group in inactive_conditional_sweep_df.groupby(
        ["anchor_id", "theta_index"]
    ):
        anchor_summaries.append({
            "anchor_id": int(anchor_id),
            "theta_index": int(theta_index),
            "conditional_p_amplitude": float(group["p_optimal"].max() - group["p_optimal"].min()),
            "conditional_p_min_relative": float(group["p_relative_to_core"].min()),
            "conditional_energy_range": float(
                group["expected_energy"].max() - group["expected_energy"].min()
            ),
            "conditional_tvd_max": float(group["tvd_vs_anchor"].max()),
            "conditional_worst_rank": int(group["dominant_valid_rank"].max()),
            "conditional_min_top10_mass": float(group["p_top_10_classical"].min()),
        })

    inactive_conditional_anchor_summary_df = pd.DataFrame(anchor_summaries)
    inactive_conditional_summary_df = inactive_conditional_anchor_summary_df.groupby(
        "theta_index", as_index=False
    ).agg(
        conditional_p_amplitude_median=("conditional_p_amplitude", "median"),
        conditional_p_min_relative_median=("conditional_p_min_relative", "median"),
        conditional_energy_range_median=("conditional_energy_range", "median"),
        conditional_tvd_max_median=("conditional_tvd_max", "median"),
        conditional_worst_rank_median=("conditional_worst_rank", "median"),
        conditional_min_top10_mass_median=("conditional_min_top10_mass", "median"),
    ).merge(
        parameter_map_df,
        on="theta_index",
        how="left",
        validate="one_to_one",
    ).sort_values("conditional_p_amplitude_median", ascending=False)

    display(inactive_conditional_summary_df[[
        "theta_index",
        "ansatz_gate_type",
        "logical_assets",
        "conditional_p_amplitude_median",
        "conditional_p_min_relative_median",
        "conditional_energy_range_median",
        "conditional_tvd_max_median",
        "conditional_worst_rank_median",
        "conditional_min_top10_mass_median",
    ]])

    fig, ax = plt.subplots(figsize=(12, 5))
    ordered = inactive_conditional_summary_df.sort_values(
        "conditional_p_amplitude_median", ascending=False
    )
    ax.bar(ordered["theta_index"].astype(str), ordered["conditional_p_amplitude_median"])
    ax.set_xlabel("theta inativo, com os sete ativos fixos nos picos")
    ax.set_ylabel("amplitude condicional mediana de P(x*)")
    ax.set_title("Os parâmetros inativos continuam inativos quando o núcleo ativo é fixado?")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "inactive_conditional_amplitude_active_core_fixed.png", dpi=180)
    plt.show()


#### Resposta fornecida pelas varreduras condicionais

Os 23 parâmetros permaneceram planos mesmo depois de fixar os sete ativos nos picos. As maiores amplitudes observadas ficaram próximas de \(10^{-16}\).

Isso rejeita, para os observáveis atuais, a hipótese de que os parâmetros inativos esconderiam uma influência individual ativada somente após reposicionar o núcleo.


### Célula 20 — Perturbação aleatória acumulada dos 23 inativos

**Em termos simples:** os parâmetros inativos são embaralhados em uma ordem aleatória reproduzível. Depois, eles são modificados progressivamente: primeiro 1, depois 2, até os 23.

Cada caminho parte do núcleo ativo e usa alvos angulares sorteados dentro do período correto de cada parâmetro.

Para cada quantidade $m$ de parâmetros alterados, a célula resume a razão

$$
R_m=\frac{P_m(x^*)}{P_{\mathrm{núcleo}}(x^*)}
$$

e também gap de energia, TVD, ranking dominante e massa nos 10 melhores portfólios. Os quantis de 5% e 95% mostram a dispersão entre ordens e alvos aleatórios.


In [ ]:
# ============================================================
# 20. PERTURBAÇÃO ALEATÓRIA ANINHADA: 0, 1, 2, ..., 23 INATIVOS
# ============================================================


def run_inactive_nested_path(anchor_row, repeat_index):
    """Altera progressivamente 0, 1, ..., 23 inativos numa ordem aleatória fixa."""
    anchor_id = int(anchor_row["anchor_id"])
    repeat_index = int(repeat_index)
    path = CHECKPOINT_DIR / (
        f"v204_inactive_nested_anchor_{anchor_id:02d}_repeat_{repeat_index:03d}.pkl"
    )
    if path.exists():
        return pd.read_pickle(path)

    # A semente incorpora âncora e repetição, tornando cada caminho reproduzível.
    rng = np.random.default_rng(
        RANDOM_SEED + 9_000_000 + 10_000 * anchor_id + repeat_index
    )
    theta_core = active_core_theta_by_anchor[anchor_id].copy()
    core_metrics = evaluate_theta(theta_core)
    reference_probability = core_metrics["full_probability"]

    # A ordem e o alvo de cada theta são sorteados uma vez e reutilizados no caminho.
    random_order = tuple(int(value) for value in rng.permutation(INACTIVE_SET))
    random_targets = {
        theta_index: float(rng.uniform(0.0, parameter_period(theta_index)))
        for theta_index in random_order
    }

    theta_test = theta_core.copy()
    rows = []
    # O vetor é cumulativo: a modificação anterior permanece quando a próxima entra.
    for subset_size in range(0, len(INACTIVE_SET) + 1):
        if subset_size > 0:
            new_index = random_order[subset_size - 1]
            theta_test[new_index] = random_targets[new_index]

        metrics = evaluate_theta(theta_test, reference_probability=reference_probability)
        metrics.pop("full_probability")
        rows.append({
            "anchor_id": anchor_id,
            "repeat_index": repeat_index,
            "subset_size": int(subset_size),
            "changed_indices": tuple(random_order[:subset_size]),
            "last_added_theta": (
                int(random_order[subset_size - 1]) if subset_size > 0 else None
            ),
            "random_order": random_order,
            "random_targets": random_targets,
            "p_relative_to_core": float(
                metrics["p_optimal"] / max(core_metrics["p_optimal"], 1e-15)
            ),
            **metrics,
        })

    frame = pd.DataFrame(rows)
    frame.to_pickle(path)
    return frame


inactive_nested_random_df = pd.DataFrame()
inactive_nested_summary_df = pd.DataFrame()
if RUN_ACTIVE_CORE_TESTS:
    frames = []
    for _, anchor_row in anchors_df.iterrows():
        for repeat_index in range(INACTIVE_NESTED_REPEATS):
            frames.append(run_inactive_nested_path(anchor_row, repeat_index))
        print(f"trajetórias aninhadas concluídas para âncora {int(anchor_row['anchor_id'])}")
    inactive_nested_random_df = pd.concat(frames, ignore_index=True)

    # Agrega todas as âncoras e repetições por quantidade de inativos modificados.
    inactive_nested_summary_df = inactive_nested_random_df.groupby(
        "subset_size", as_index=False
    ).agg(
        p_relative_median=("p_relative_to_core", "median"),
        p_relative_q05=("p_relative_to_core", lambda s: float(s.quantile(0.05))),
        p_relative_q95=("p_relative_to_core", lambda s: float(s.quantile(0.95))),
        p_relative_min=("p_relative_to_core", "min"),
        energy_gap_median=("energy_gap", "median"),
        energy_gap_q95=("energy_gap", lambda s: float(s.quantile(0.95))),
        tvd_median=("tvd_vs_anchor", "median"),
        tvd_q95=("tvd_vs_anchor", lambda s: float(s.quantile(0.95))),
        dominant_rank_median=("dominant_valid_rank", "median"),
        dominant_rank_q95=("dominant_valid_rank", lambda s: float(s.quantile(0.95))),
        top10_mass_median=("p_top_10_classical", "median"),
        top10_mass_q05=("p_top_10_classical", lambda s: float(s.quantile(0.05))),
    )

    display(inactive_nested_summary_df)

    fig, ax = plt.subplots(figsize=(10, 5))
    x = inactive_nested_summary_df["subset_size"].to_numpy()
    # Mostra o desvio em torno de 1, evitando o offset automático do Matplotlib.
    median_delta = inactive_nested_summary_df["p_relative_median"].to_numpy() - 1.0
    q05_delta = inactive_nested_summary_df["p_relative_q05"].to_numpy() - 1.0
    q95_delta = inactive_nested_summary_df["p_relative_q95"].to_numpy() - 1.0
    ax.plot(x, median_delta, marker="o", label="mediana - 1")
    ax.fill_between(x, q05_delta, q95_delta, alpha=0.25, label="faixa 5%–95%")
    ax.axhline(0.0, linestyle="--", linewidth=1.2)
    ax.set_xlabel("quantidade de parâmetros inativos modificados")
    ax.set_ylabel("P(x*) / P(x*) do núcleo ativo - 1")
    ax.set_title("Desvio causado ao modificar progressivamente os 23 inativos")
    ax.set_xticks(range(0, len(INACTIVE_SET) + 1))
    ax.ticklabel_format(axis="y", style="scientific", scilimits=(-3, 3), useOffset=False)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "inactive_nested_random_0_to_23_delta_from_one.png", dpi=180)
    plt.show()


#### Resposta fornecida pela perturbação de 1 até 23 inativos

Na execução anterior, modificar progressivamente todos os 23 parâmetros manteve a razão \(P(x^*)/P_{\rm núcleo}\) igual a 1 dentro da precisão numérica.

O gráfico desta versão mostra diretamente

\[
\frac{P(x^*)}{P_{\rm núcleo}(x^*)}-1,
\]

evitando que o offset automático do Matplotlib transforme variações da ordem de \(10^{-12}\) em uma falsa queda visual.


### Célula 21 — Teste de resgate do núcleo ativo

**Em termos simples:** primeiro todos os 30 parâmetros recebem valores aleatórios. Em seguida, somente os sete ativos são restaurados para os valores do núcleo; os outros 23 continuam aleatórios.

A comparação direta é:

$$
\theta_{\mathrm{todos\ aleatórios}}
\quad\longrightarrow\quad
\theta_{\mathrm{23\ aleatórios+7\ ativos}}.
$$

Se a segunda configuração recuperar grande parte de $P(x^*)$, isso fornece evidência de suficiência do núcleo ativo para este Hamiltoniano. O teste é repetido várias vezes e resumido por mediana e quantis.


In [ ]:
# ============================================================
# 21. TESTE DE RESGATE: TODOS ALEATÓRIOS, DEPOIS FIXAR SÓ OS 7 ATIVOS
# ============================================================


def run_active_core_rescue(anchor_row, trial_index):
    """Compara 30 aleatórios com o mesmo vetor após restaurar apenas os 7 ativos."""
    anchor_id = int(anchor_row["anchor_id"])
    trial_index = int(trial_index)
    path = CHECKPOINT_DIR / (
        f"v204_active_core_rescue_anchor_{anchor_id:02d}_trial_{trial_index:04d}.pkl"
    )
    if path.exists():
        return pd.read_pickle(path)

    rng = np.random.default_rng(
        RANDOM_SEED + 12_000_000 + 10_000 * anchor_id + trial_index
    )
    theta_core = active_core_theta_by_anchor[anchor_id].copy()
    core_metrics = evaluate_theta(theta_core)
    reference_probability = core_metrics["full_probability"]

    # Fundo totalmente aleatório, respeitando o período físico de cada parâmetro.
    theta_all_random = np.asarray([
        rng.uniform(0.0, parameter_period(theta_index))
        for theta_index in range(N_PARAMETERS)
    ], dtype=float)
    random_metrics = evaluate_theta(
        theta_all_random,
        reference_probability=reference_probability,
    )

    # Intervenção de resgate: somente os índices ativos são substituídos.
    theta_rescued = theta_all_random.copy()
    for theta_index in ACTIVE_SET:
        theta_rescued[theta_index] = theta_core[theta_index]
    rescued_metrics = evaluate_theta(
        theta_rescued,
        reference_probability=reference_probability,
    )

    row = {
        "anchor_id": anchor_id,
        "trial_index": trial_index,
        "p_core": float(core_metrics["p_optimal"]),
        "p_all_random": float(random_metrics["p_optimal"]),
        "p_active_core_rescue": float(rescued_metrics["p_optimal"]),
        "random_relative_to_core": float(
            random_metrics["p_optimal"] / max(core_metrics["p_optimal"], 1e-15)
        ),
        "rescue_relative_to_core": float(
            rescued_metrics["p_optimal"] / max(core_metrics["p_optimal"], 1e-15)
        ),
        "energy_gap_all_random": float(random_metrics["energy_gap"]),
        "energy_gap_active_core_rescue": float(rescued_metrics["energy_gap"]),
        "tvd_all_random_vs_core": float(random_metrics["tvd_vs_anchor"]),
        "tvd_rescue_vs_core": float(rescued_metrics["tvd_vs_anchor"]),
        "rank_all_random": int(random_metrics["dominant_valid_rank"]),
        "rank_active_core_rescue": int(rescued_metrics["dominant_valid_rank"]),
        "top10_all_random": float(random_metrics["p_top_10_classical"]),
        "top10_active_core_rescue": float(rescued_metrics["p_top_10_classical"]),
    }
    frame = pd.DataFrame([row])
    frame.to_pickle(path)
    return frame


active_core_rescue_df = pd.DataFrame()
active_core_rescue_summary_df = pd.DataFrame()
if RUN_ACTIVE_CORE_TESTS:
    frames = []
    for _, anchor_row in anchors_df.iterrows():
        for trial_index in range(ACTIVE_CORE_RESCUE_TRIALS):
            frames.append(run_active_core_rescue(anchor_row, trial_index))
        print(f"teste de resgate concluído para âncora {int(anchor_row['anchor_id'])}")
    active_core_rescue_df = pd.concat(frames, ignore_index=True)

    # Resume a distribuição de resultados dos ensaios por mediana e quantis.
    summary_rows = []
    for metric in [
        "random_relative_to_core",
        "rescue_relative_to_core",
        "energy_gap_all_random",
        "energy_gap_active_core_rescue",
        "tvd_all_random_vs_core",
        "tvd_rescue_vs_core",
        "rank_all_random",
        "rank_active_core_rescue",
        "top10_all_random",
        "top10_active_core_rescue",
    ]:
        values = active_core_rescue_df[metric]
        summary_rows.append({
            "metric": metric,
            "median": float(values.median()),
            "q05": float(values.quantile(0.05)),
            "q95": float(values.quantile(0.95)),
            "minimum": float(values.min()),
            "maximum": float(values.max()),
        })
    active_core_rescue_summary_df = pd.DataFrame(summary_rows)
    display(active_core_rescue_summary_df)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.boxplot(
        [
            active_core_rescue_df["random_relative_to_core"],
            active_core_rescue_df["rescue_relative_to_core"],
        ],
        showfliers=False,
    )
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["30 aleatórios", "7 ativos restaurados"])
    ax.set_ylabel("P(x*) / P(x*) do núcleo ativo")
    ax.set_title("O núcleo ativo recupera a solução com 23 parâmetros aleatórios?")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "active_core_rescue_random_all_vs_seven_fixed.png", dpi=180)
    plt.show()


#### Resposta fornecida pelo teste de resgate

Na execução já observada, aleatorizar os 30 parâmetros destruiu a solução, mas restaurar somente os sete ativos recuperou:

- \(P(x^*)\) relativo igual a 1;
- rank dominante igual a 1;
- massa top-10 igual a 1;
- TVD e gap no nível numérico.

Isso sustenta a suficiência observável do núcleo ativo para esta instância. A nova parte de statevector verificará se os 23 restantes também são nulos nas fases complexas.


# Parte VIII — statevector complexo e QFIM

Os testes anteriores mostraram que os 23 parâmetros restantes não alteram probabilidades, energia nem o ranking dos portfólios quando o núcleo ativo está fixado. Isso ainda deixa uma pergunta:

> esses parâmetros são realmente nulos no estado quântico ou alteram somente fases invisíveis ao Hamiltoniano diagonal?

Esta parte responde em duas etapas:

1. compara diretamente os statevectors por fidelidade e distância após remover a fase global;
2. calcula a QFIM para estimar quantas direções independentes realmente modificam o estado.


### Célula 22 — Fidelidade com os 23 parâmetros inativos aleatórios

Para cada âncora, o núcleo ativo é mantido fixo e os 23 parâmetros restantes recebem valores aleatórios. São comparados:

\[
F=|\langle\psi_{\rm núcleo}|\psi_{\rm teste}\rangle|^2,
\]

a distância após alinhar a fase global e a TVD entre as probabilidades.

- \(F\approx1\) e distância próxima de zero: o estado inteiro é preservado;
- \(F<1\) com TVD próxima de zero: as probabilidades são iguais, mas existem fases relativas diferentes.


In [ ]:
# ============================================================
# 22. FIDELIDADE DO STATEVECTOR COM OS 23 INATIVOS ALEATÓRIOS
# ============================================================


def compare_statevectors(reference_state, candidate_state):
    """Compara dois estados, removendo explicitamente a fase global."""
    reference_state = np.asarray(reference_state, dtype=np.complex128)
    candidate_state = np.asarray(candidate_state, dtype=np.complex128)

    reference_state = reference_state / np.linalg.norm(reference_state)
    candidate_state = candidate_state / np.linalg.norm(candidate_state)

    overlap = np.vdot(reference_state, candidate_state)
    fidelity = float(np.clip(np.abs(overlap) ** 2, 0.0, 1.0))

    # Alinha a fase global do candidato com a referência.
    phase = float(np.angle(overlap))
    aligned_candidate = candidate_state * np.exp(-1j * phase)
    phase_aligned_distance = float(
        np.linalg.norm(reference_state - aligned_candidate)
    )

    reference_probability = np.abs(reference_state) ** 2
    candidate_probability = np.abs(candidate_state) ** 2
    probability_tvd = total_variation_distance(
        reference_probability,
        candidate_probability,
    )

    # Diferença de fase apenas onde ambos os estados têm suporte relevante.
    support_mask = (
        (reference_probability > STATEVECTOR_SUPPORT_EPSILON)
        & (candidate_probability > STATEVECTOR_SUPPORT_EPSILON)
    )
    if np.any(support_mask):
        relative_phase = np.angle(
            aligned_candidate[support_mask]
            * np.conj(reference_state[support_mask])
        )
        weights = reference_probability[support_mask]
        weights = weights / weights.sum()
        phase_rms_on_support = float(
            np.sqrt(np.sum(weights * relative_phase ** 2))
        )
    else:
        phase_rms_on_support = np.nan

    return {
        "fidelity": fidelity,
        "infidelity": float(max(0.0, 1.0 - fidelity)),
        "global_phase_removed": phase,
        "phase_aligned_state_distance": phase_aligned_distance,
        "probability_tvd": probability_tvd,
        "phase_rms_on_support": phase_rms_on_support,
    }


def run_inactive_statevector_trial(anchor_row, trial_index):
    """Aleatoriza apenas os 23 inativos e compara o estado com o núcleo ativo."""
    anchor_id = int(anchor_row["anchor_id"])
    trial_index = int(trial_index)
    path = CHECKPOINT_DIR / (
        f"v204_statevector_inactive_anchor_{anchor_id:02d}_trial_{trial_index:04d}.pkl"
    )
    if path.exists():
        return pd.read_pickle(path)

    rng = np.random.default_rng(
        RANDOM_SEED + 15_000_000 + 10_000 * anchor_id + trial_index
    )
    theta_core = active_core_theta_by_anchor[anchor_id].copy()
    theta_test = theta_core.copy()

    for theta_index in INACTIVE_SET:
        theta_test[theta_index] = rng.uniform(
            0.0,
            parameter_period(theta_index),
        )

    diagnostics = compare_statevectors(
        statevector_from_theta(theta_core),
        statevector_from_theta(theta_test),
    )

    frame = pd.DataFrame([{
        "anchor_id": anchor_id,
        "trial_index": trial_index,
        "changed_indices": INACTIVE_SET,
        **diagnostics,
    }])
    frame.to_pickle(path)
    return frame


statevector_inactive_df = pd.DataFrame()
statevector_inactive_summary_df = pd.DataFrame()

if RUN_STATEVECTOR_PHASE_TESTS:
    frames = []
    for _, anchor_row in anchors_df.iterrows():
        for trial_index in range(STATEVECTOR_PHASE_TRIALS):
            frames.append(
                run_inactive_statevector_trial(anchor_row, trial_index)
            )
        print(
            f"statevector concluído para âncora "
            f"{int(anchor_row['anchor_id'])}"
        )

    statevector_inactive_df = pd.concat(frames, ignore_index=True)

    summary_rows = []
    for metric in [
        "fidelity",
        "infidelity",
        "phase_aligned_state_distance",
        "probability_tvd",
        "phase_rms_on_support",
    ]:
        values = statevector_inactive_df[metric].dropna()
        summary_rows.append({
            "metric": metric,
            "median": float(values.median()),
            "q05": float(values.quantile(0.05)),
            "q95": float(values.quantile(0.95)),
            "minimum": float(values.min()),
            "maximum": float(values.max()),
        })
    statevector_inactive_summary_df = pd.DataFrame(summary_rows)
    display(statevector_inactive_summary_df)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    plotted_metrics = [
        ("infidelity", "1 - fidelidade"),
        (
            "phase_aligned_state_distance",
            "distância após remover fase global",
        ),
        ("probability_tvd", "TVD das probabilidades"),
    ]
    numeric_floor = 1e-18
    for ax, (metric, label) in zip(axes, plotted_metrics):
        values = np.maximum(
            statevector_inactive_df[metric].to_numpy(dtype=float),
            numeric_floor,
        )
        ax.boxplot(values, showfliers=False)
        ax.set_yscale("log")
        ax.set_xticks([1])
        ax.set_xticklabels([label], rotation=12)
        ax.set_title(label)
    fig.suptitle(
        "Os 23 parâmetros inativos alteram o statevector complexo?"
    )
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / "inactive_statevector_fidelity_phase_probability.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()

    fidelity_median = float(
        statevector_inactive_df["fidelity"].median()
    )
    tvd_median = float(
        statevector_inactive_df["probability_tvd"].median()
    )
    distance_median = float(
        statevector_inactive_df[
            "phase_aligned_state_distance"
        ].median()
    )

    if (
        fidelity_median >= 1.0 - 1e-10
        and distance_median <= 1e-8
    ):
        conclusion = (
            "Os 23 parâmetros são numericamente nulos também para o "
            "statevector, salvo fase global."
        )
    elif (
        tvd_median <= 1e-10
        and fidelity_median < 1.0 - 1e-8
    ):
        conclusion = (
            "As probabilidades permanecem iguais, mas os 23 parâmetros "
            "alteram fases relativas do estado."
        )
    else:
        conclusion = (
            "Os 23 parâmetros alteram o estado e parte do efeito também "
            "aparece nas probabilidades; revisar a classificação."
        )
    display(Markdown(f"**Diagnóstico automático:** {conclusion}"))


#### Como interpretar o resultado

Nas execuções anteriores, a TVD permaneceu no nível numérico mesmo com os 23 parâmetros aleatórios. Esta nova célula verifica se essa invariância também vale para as amplitudes complexas.

A fidelidade deve ser interpretada junto com a TVD:

- fidelidade e TVD perfeitas: redundância completa para o estado atual;
- TVD perfeita e fidelidade menor: parâmetros de fase, invisíveis à leitura computacional;
- ambas diferentes: a classificação anterior dependia do ponto de referência.


### Célula 23 — QFIM e dimensão geométrica efetiva

A QFIM mede quanto o estado muda sob pequenas alterações dos parâmetros:

\[
(F_Q)_{ij}
=
4\,\mathrm{Re}
\left[
\langle\partial_i\psi|\partial_j\psi\rangle
-
\langle\partial_i\psi|\psi\rangle
\langle\psi|\partial_j\psi\rangle
\right].
\]

A projeção do segundo termo remove a direção de fase global. O número de autovalores significativamente diferentes de zero fornece o rank local da QFIM.


In [ ]:
# ============================================================
# 23. QFIM POR DIFERENÇAS CENTRAIS
# ============================================================


def pure_state_qfim(theta, step=QFIM_FINITE_DIFFERENCE_STEP):
    """Calcula a QFIM de um estado puro e seus diagnósticos espectrais."""
    theta = np.asarray(theta, dtype=float).reshape(-1)
    psi = statevector_from_theta(theta)
    psi = psi / np.linalg.norm(psi)

    derivatives = np.empty(
        (N_PARAMETERS, len(psi)),
        dtype=np.complex128,
    )

    # Cada linha contém |∂_j psi>, estimado sem otimização.
    for theta_index in range(N_PARAMETERS):
        theta_plus = theta.copy()
        theta_minus = theta.copy()
        theta_plus[theta_index] += step
        theta_minus[theta_index] -= step

        derivatives[theta_index] = (
            statevector_from_theta(theta_plus)
            - statevector_from_theta(theta_minus)
        ) / (2.0 * step)

    gram = derivatives.conj() @ derivatives.T
    overlap_with_state = derivatives.conj() @ psi
    projected_gram = gram - np.outer(
        overlap_with_state,
        overlap_with_state.conj(),
    )
    qfim = 4.0 * np.real(projected_gram)
    qfim = 0.5 * (qfim + qfim.T)

    raw_eigenvalues = np.linalg.eigvalsh(qfim)
    raw_eigenvalues = np.sort(raw_eigenvalues)[::-1]
    eigenvalues = np.where(
        raw_eigenvalues > -QFIM_ABSOLUTE_EIGEN_THRESHOLD,
        np.maximum(raw_eigenvalues, 0.0),
        raw_eigenvalues,
    )

    largest = max(float(np.max(eigenvalues)), 0.0)
    threshold = max(
        QFIM_ABSOLUTE_EIGEN_THRESHOLD,
        QFIM_RELATIVE_EIGEN_THRESHOLD * largest,
    )
    numerical_rank = int(np.sum(eigenvalues > threshold))

    positive = eigenvalues[eigenvalues > threshold]
    if positive.size:
        weights = positive / positive.sum()
        effective_rank = float(
            np.exp(-np.sum(weights * np.log(weights)))
        )
    else:
        effective_rank = 0.0

    return {
        "qfim": qfim,
        "eigenvalues": eigenvalues,
        "diagonal": np.diag(qfim).copy(),
        "numerical_threshold": threshold,
        "numerical_rank": numerical_rank,
        "effective_rank": effective_rank,
        "trace": float(np.trace(qfim)),
        "minimum_raw_eigenvalue": float(raw_eigenvalues.min()),
    }


qfim_anchor_rows = []
qfim_spectrum_rows = []
qfim_diagonal_rows = []
qfim_matrices = []

if RUN_QFIM_TESTS:
    qfim_anchors = anchors_df.head(
        min(QFIM_MAX_ANCHORS, len(anchors_df))
    )

    for _, anchor_row in qfim_anchors.iterrows():
        anchor_id = int(anchor_row["anchor_id"])
        path = CHECKPOINT_DIR / (
            f"v204_qfim_active_core_anchor_{anchor_id:02d}.npz"
        )

        if path.exists():
            saved = np.load(path)
            result = {
                "qfim": np.asarray(saved["qfim"], dtype=float),
                "eigenvalues": np.asarray(
                    saved["eigenvalues"], dtype=float
                ),
                "diagonal": np.asarray(saved["diagonal"], dtype=float),
                "numerical_threshold": float(
                    saved["numerical_threshold"]
                ),
                "numerical_rank": int(saved["numerical_rank"]),
                "effective_rank": float(saved["effective_rank"]),
                "trace": float(saved["trace"]),
                "minimum_raw_eigenvalue": float(
                    saved["minimum_raw_eigenvalue"]
                ),
            }
        else:
            result = pure_state_qfim(
                active_core_theta_by_anchor[anchor_id]
            )
            np.savez_compressed(path, **result)

        qfim = result["qfim"]
        eigenvalues = result["eigenvalues"]
        diagonal = result["diagonal"]
        qfim_matrices.append(qfim)

        active_sum = float(np.sum(diagonal[list(ACTIVE_SET)]))
        inactive_sum = float(
            np.sum(diagonal[list(INACTIVE_SET)])
        )

        qfim_anchor_rows.append({
            "anchor_id": anchor_id,
            "numerical_rank": int(result["numerical_rank"]),
            "effective_rank": float(result["effective_rank"]),
            "trace": float(result["trace"]),
            "numerical_threshold": float(
                result["numerical_threshold"]
            ),
            "minimum_raw_eigenvalue": float(
                result["minimum_raw_eigenvalue"]
            ),
            "active_diagonal_sum": active_sum,
            "inactive_diagonal_sum": inactive_sum,
            "active_fraction_of_diagonal": float(
                active_sum / max(active_sum + inactive_sum, 1e-30)
            ),
        })

        for eigen_index, value in enumerate(eigenvalues):
            qfim_spectrum_rows.append({
                "anchor_id": anchor_id,
                "eigen_index": int(eigen_index),
                "eigenvalue": float(value),
                "normalized_eigenvalue": float(
                    value / max(eigenvalues[0], 1e-30)
                ),
            })

        for theta_index, value in enumerate(diagonal):
            qfim_diagonal_rows.append({
                "anchor_id": anchor_id,
                "theta_index": int(theta_index),
                "is_active": bool(theta_index in ACTIVE_SET),
                "qfim_diagonal": float(value),
            })

        print(
            f"QFIM âncora {anchor_id}: "
            f"rank={result['numerical_rank']}, "
            f"rank efetivo={result['effective_rank']:.3f}"
        )

qfim_anchor_summary_df = pd.DataFrame(qfim_anchor_rows)
qfim_spectrum_df = pd.DataFrame(qfim_spectrum_rows)
qfim_diagonal_df = pd.DataFrame(qfim_diagonal_rows)
qfim_diagonal_summary_df = pd.DataFrame()

if not qfim_anchor_summary_df.empty:
    display(qfim_anchor_summary_df)

    qfim_diagonal_summary_df = qfim_diagonal_df.groupby(
        ["theta_index", "is_active"],
        as_index=False,
    ).agg(
        qfim_diagonal_median=("qfim_diagonal", "median"),
        qfim_diagonal_q05=(
            "qfim_diagonal",
            lambda s: float(s.quantile(0.05)),
        ),
        qfim_diagonal_q95=(
            "qfim_diagonal",
            lambda s: float(s.quantile(0.95)),
        ),
    )
    display(qfim_diagonal_summary_df)

    spectrum_summary = qfim_spectrum_df.groupby(
        "eigen_index",
        as_index=False,
    ).agg(
        median=("normalized_eigenvalue", "median"),
        q05=(
            "normalized_eigenvalue",
            lambda s: float(s.quantile(0.05)),
        ),
        q95=(
            "normalized_eigenvalue",
            lambda s: float(s.quantile(0.95)),
        ),
    )

    fig, ax = plt.subplots(figsize=(9, 5))
    x = spectrum_summary["eigen_index"].to_numpy() + 1
    floor = 1e-18
    median = np.maximum(
        spectrum_summary["median"].to_numpy(), floor
    )
    q05 = np.maximum(
        spectrum_summary["q05"].to_numpy(), floor
    )
    q95 = np.maximum(
        spectrum_summary["q95"].to_numpy(), floor
    )
    ax.semilogy(x, median, marker="o", label="mediana")
    ax.fill_between(
        x, q05, q95, alpha=0.25, label="faixa 5%–95%"
    )
    ax.axhline(
        QFIM_RELATIVE_EIGEN_THRESHOLD,
        linestyle="--",
        linewidth=1.2,
        label="limiar relativo",
    )
    ax.set_xlabel("índice do autovalor")
    ax.set_ylabel("autovalor / maior autovalor")
    ax.set_title("Espectro da QFIM nas âncoras")
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / "qfim_normalized_eigenvalue_spectrum.png",
        dpi=180,
    )
    plt.show()

    ordered = qfim_diagonal_summary_df.sort_values(
        "theta_index"
    )
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(
        ordered["theta_index"].astype(str),
        np.maximum(
            ordered["qfim_diagonal_median"].to_numpy(),
            1e-18,
        ),
    )
    ax.set_yscale("log")
    ax.set_xlabel("theta_index")
    ax.set_ylabel("diagonal mediana da QFIM")
    ax.set_title(
        "Sensibilidade geométrica local de cada parâmetro"
    )
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / "qfim_parameter_diagonal_median.png",
        dpi=180,
    )
    plt.show()

    median_qfim = np.median(
        np.stack(qfim_matrices), axis=0
    )
    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(
        median_qfim,
        origin="lower",
        aspect="auto",
    )
    ax.set_xlabel("theta_j")
    ax.set_ylabel("theta_i")
    ax.set_title("QFIM mediana entre âncoras")
    fig.colorbar(image, ax=ax, label="(F_Q)ij")
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / "qfim_median_matrix.png",
        dpi=180,
    )
    plt.show()

    median_rank = float(
        qfim_anchor_summary_df["numerical_rank"].median()
    )
    median_active_fraction = float(
        qfim_anchor_summary_df[
            "active_fraction_of_diagonal"
        ].median()
    )
    display(Markdown(
        "**Diagnóstico automático:** rank mediano da QFIM = "
        f"**{median_rank:.1f}**; fração mediana da diagonal "
        "concentrada nos sete ativos = "
        f"**{median_active_fraction:.6f}**."
    ))


#### Como interpretar a QFIM

A varredura mede a dimensão **observável** em probabilidade e energia. A QFIM mede a dimensão **geométrica local** do estado completo.

- rank próximo de 7: os sete parâmetros correspondem aproximadamente às direções físicas independentes;
- rank maior que 7: existem direções adicionais, possivelmente de fase;
- rank menor que 7: alguns dos sete parâmetros ativos são redundantes entre si.

A diagonal ajuda a localizar parâmetros sensíveis, mas o rank depende da matriz completa, inclusive das correlações fora da diagonal.


# Parte IX — perturbações acumuladas dos melhores, controles e gate-off

As varreduras multidimensionais completas crescem exponencialmente. Para comparar um, dois, três e quatro parâmetros, usamos uma trajetória controlada:

$$
\theta_j(\lambda)=\theta_j^{\mathrm{âncora}}+
\lambda\,\Delta_j,
\qquad \lambda\in[0,1].
$$

São executados dois alvos:

1. `zero`: leva os parâmetros selecionados ao ângulo zero pelo menor deslocamento circular;
2. `individual_worst`: leva cada parâmetro ao pior ângulo encontrado na sua própria varredura, para aquela âncora.

Os subconjuntos são:

- fortes: `[17]`, `[17,2]`, `[17,2,14]`, `[17,2,14,3]`;
- controles: `[24]`, `[24,0]`, `[24,0,1]`, `[24,0,1,9]`;
- gate-off: `[25]`, `[25,27]`.


### Célula 24 — Perturbações acumuladas dirigidas

**Em termos simples:** esta célula move grupos de parâmetros desde a âncora até um alvo conhecido, usando uma trajetória contínua controlada por $\lambda$:

$$
\theta_j(\lambda)
=
\theta_j^{(a)}+\lambda\,\Delta_j,
\qquad 0\leq\lambda\leq1.
$$

O deslocamento $\Delta_j$ segue o menor caminho circular. Os alvos são:

- `zero`: ângulo zero;
- `individual_worst`: pior ângulo observado na varredura individual.

As famílias `strong`, `control` e `gate_off` permitem comparar grupos de parâmetros fortes, controles e desligamento de portas.


In [ ]:
# ============================================================
# 24. VARREDURAS ACUMULADAS
# ============================================================


def nested_subsets(order):
    """Gera prefixos cumulativos: [a], [a,b], [a,b,c], ..."""
    return [tuple(order[:size]) for size in range(1, len(order) + 1)]


# Define famílias comparáveis de parâmetros fortes, controles e gate-off.
subset_definitions = []
for family, order in [
    ("strong", STRONG_ORDER),
    ("control", CONTROL_ORDER),
    ("gate_off", GATE_OFF_ORDER),
]:
    for subset in nested_subsets(order):
        subset_definitions.append({"family": family, "subset": subset})


def worst_angle_lookup(anchor_id, theta_index):
    """Busca o pior ângulo primeiro na varredura detalhada e depois no atlas."""
    group = individual_sweep_df.loc[
        individual_sweep_df["anchor_id"].eq(int(anchor_id))
        & individual_sweep_df["theta_index"].eq(int(theta_index))
    ]
    if not group.empty:
        return float(group.loc[group["p_optimal"].idxmin(), "theta_value"])

    atlas_group = atlas_sweep_df.loc[
        atlas_sweep_df["anchor_id"].eq(int(anchor_id))
        & atlas_sweep_df["theta_index"].eq(int(theta_index))
    ]
    if not atlas_group.empty:
        return float(atlas_group.loc[atlas_group["p_optimal"].idxmin(), "theta_value"])

    raise KeyError(f"Nenhuma varredura disponível para theta_{theta_index}, âncora {anchor_id}.")


def run_cumulative_task(anchor_row, family, subset, target_kind):
    """Move simultaneamente um subconjunto ao alvo seguindo o menor arco angular."""
    anchor_id = int(anchor_row["anchor_id"])
    subset = tuple(int(value) for value in subset)
    stem = "_".join(map(str, subset))
    path = CHECKPOINT_DIR / f"cumulative_{family}_{target_kind}_anchor_{anchor_id:02d}_{stem}.pkl"
    if path.exists():
        return pd.read_pickle(path)

    theta_anchor = np.asarray(anchor_row["theta_vector"], dtype=float).copy()
    anchor_metrics = evaluate_theta(theta_anchor)
    reference_probability = anchor_metrics["full_probability"]
    # lambda=0 reproduz a âncora e lambda=1 alcança o alvo final.
    lambdas = np.linspace(0.0, 1.0, CUMULATIVE_LAMBDA_POINTS)

    targets = {}
    for theta_index in subset:
        period = parameter_period(theta_index)
        if target_kind == "zero":
            target = 0.0
        elif target_kind == "individual_worst":
            target = worst_angle_lookup(anchor_id, theta_index)
        else:
            raise ValueError(target_kind)
        targets[theta_index] = (target, period)

    rows = []
    for lambda_index, lambda_value in enumerate(lambdas):
        theta_test = theta_anchor.copy()
        # Cada componente percorre o menor deslocamento circular até seu alvo.
        for theta_index, (target, period) in targets.items():
            delta = shortest_delta_to_target(theta_anchor[theta_index], target, period)
            theta_test[theta_index] = theta_anchor[theta_index] + lambda_value * delta
        metrics = evaluate_theta(theta_test, reference_probability=reference_probability)
        metrics.pop("full_probability")
        rows.append({
            "anchor_id": anchor_id,
            "family": family,
            "subset": subset,
            "subset_label": "+".join(f"theta_{value}" for value in subset),
            "subset_size": len(subset),
            "target_kind": target_kind,
            "lambda_index": lambda_index,
            "lambda": float(lambda_value),
            **metrics,
        })

    frame = pd.DataFrame(rows)
    frame.to_pickle(path)
    return frame


cumulative_frames = []
for _, anchor_row in anchors_df.iterrows():
    for definition in subset_definitions:
        for target_kind in ("zero", "individual_worst"):
            cumulative_frames.append(run_cumulative_task(
                anchor_row,
                definition["family"],
                definition["subset"],
                target_kind,
            ))

cumulative_sweep_df = pd.concat(cumulative_frames, ignore_index=True)

# Queda relativa em relação a lambda=0 por curva.
cumulative_sweep_df["p_relative_to_anchor"] = cumulative_sweep_df.groupby(
    ["anchor_id", "family", "subset_label", "target_kind"]
)["p_optimal"].transform(lambda series: series / max(float(series.iloc[0]), 1e-15))

cumulative_endpoint_df = cumulative_sweep_df.loc[
    cumulative_sweep_df["lambda"].eq(1.0)
].groupby(
    ["family", "subset_label", "subset_size", "target_kind"],
    as_index=False,
).agg(
    p_endpoint_median=("p_optimal", "median"),
    relative_p_endpoint_median=("p_relative_to_anchor", "median"),
    energy_gap_endpoint_median=("energy_gap", "median"),
    tvd_endpoint_median=("tvd_vs_anchor", "median"),
)

display(cumulative_endpoint_df.sort_values(
    ["target_kind", "family", "subset_size"]
))

for target_kind in ("zero", "individual_worst"):
    fig, ax = plt.subplots(figsize=(11, 6))
    subset = cumulative_sweep_df.loc[cumulative_sweep_df["target_kind"].eq(target_kind)]
    for (family, label), group in subset.groupby(["family", "subset_label"]):
        curve = group.groupby("lambda", as_index=False)["p_relative_to_anchor"].median()
        ax.plot(curve["lambda"], curve["p_relative_to_anchor"], label=f"{family}: {label}")
    ax.set_xlabel("lambda da perturbação acumulada")
    ax.set_ylabel("P(x*) / P(x*) da âncora")
    ax.set_title(f"Perturbação acumulada — alvo {target_kind}")
    ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"cumulative_{target_kind}.png", dpi=180)
    plt.show()


# Parte X — superfícies 2D, 21 pares ativos e separabilidade multiplicativa

Esta parte amplia o teste anterior de dois pares para todos os

\[
\binom{7}{2}=21
\]

pares do núcleo ativo. A campanha completa é resumida em duas matrizes:

1. erro relativo do modelo multiplicativo;
2. fração explicada pela melhor aproximação SVD de posto 1.

Para evitar centenas de figuras repetitivas, somente os pares indicados em `INTERACTION_PLOT_PAIRS` recebem gráficos detalhados. Todos os 21 pares permanecem nas tabelas e nos mapas-resumo.


### Célula 25 — Superfícies 2D e separabilidade dos 21 pares

Para cada par \((i,j)\), o notebook compara:

\[
F_{\mathrm{aditivo}}
=
F_i(\phi_i)+F_j(\phi_j)-F_0,
\]

\[
F_{\mathrm{multiplicativo}}
=
\frac{F_i(\phi_i)F_j(\phi_j)}{F_0}.
\]

Também é calculada a melhor aproximação de posto 1:

\[
F\approx\sigma_1u_1v_1^\mathsf{T}.
\]

O erro logarítmico agora ignora pontos abaixo do limiar de probabilidade, pois nós numéricos próximos de zero não devem dominar a métrica. Os checkpoints são salvos em `float64`, preservando resíduos no nível de precisão de máquina.


Por padrão, as superfícies são construídas sobre o **núcleo ativo**, isto é, com os sete parâmetros inicialmente posicionados em suas melhores regiões. Essa escolha mantém o teste dos pares consistente com o teste multidimensional posterior.


In [ ]:
# ============================================================
# 25. SUPERFÍCIES 2D DOS 21 PARES, MODELOS E SVD
# ============================================================


def regression_surface_metrics(actual, predicted, epsilon=SEPARABILITY_EPSILON):
    """Calcula erros absolutos, relativos e R² entre duas superfícies."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    residual = actual - predicted
    mse = float(np.mean(residual ** 2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(residual)))
    max_abs = float(np.max(np.abs(residual)))
    fro_actual = float(np.linalg.norm(actual))
    relative_fro_error = float(
        np.linalg.norm(residual) / max(fro_actual, epsilon)
    )
    centered = actual - actual.mean()
    denominator = float(np.sum(centered ** 2))
    r2 = (
        float(1.0 - np.sum(residual ** 2) / denominator)
        if denominator > epsilon
        else np.nan
    )
    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "max_abs": max_abs,
        "relative_fro_error": relative_fro_error,
        "r2": r2,
    }, residual


def rank1_svd_diagnostics(surface, epsilon=SEPARABILITY_EPSILON):
    """Obtém a melhor aproximação de posto 1 e o espectro singular."""
    surface = np.asarray(surface, dtype=float)
    u, singular_values, vh = np.linalg.svd(
        surface, full_matrices=False
    )
    rank1 = singular_values[0] * np.outer(
        u[:, 0], vh[0, :]
    )
    total_power = float(np.sum(singular_values ** 2))
    explained = float(
        singular_values[0] ** 2 / max(total_power, epsilon)
    )
    relative_error = float(
        np.linalg.norm(surface - rank1)
        / max(np.linalg.norm(surface), epsilon)
    )
    singular_probabilities = (
        singular_values / max(singular_values.sum(), epsilon)
    )
    positive = singular_probabilities[
        singular_probabilities > 0.0
    ]
    effective_rank = float(
        np.exp(-np.sum(positive * np.log(positive)))
    )
    return (
        rank1,
        singular_values,
        explained,
        relative_error,
        effective_rank,
    )


def reliable_log_surface_metrics(actual, predicted):
    """Erro logarítmico sem deixar nós numéricos dominarem a métrica."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mask = (
        (actual > LOG_PROBABILITY_THRESHOLD)
        & (predicted > LOG_PROBABILITY_THRESHOLD)
    )
    coverage = float(mask.mean())
    if not np.any(mask):
        return {
            "log_rmse_reliable": np.nan,
            "weighted_log_rmse": np.nan,
            "log_coverage": coverage,
        }

    log_residual = (
        np.log10(actual[mask])
        - np.log10(predicted[mask])
    )
    log_rmse = float(
        np.sqrt(np.mean(log_residual ** 2))
    )
    weights = actual[mask]
    weights = weights / weights.sum()
    weighted_log_rmse = float(
        np.sqrt(np.sum(weights * log_residual ** 2))
    )
    return {
        "log_rmse_reliable": log_rmse,
        "weighted_log_rmse": weighted_log_rmse,
        "log_coverage": coverage,
    }


def pair_is_selected_for_plot(theta_i, theta_j):
    """Evita gerar centenas de figuras durante a campanha dos 21 pares."""
    pair = tuple(sorted((int(theta_i), int(theta_j))))
    selected = {
        tuple(sorted(map(int, pair_value)))
        for pair_value in INTERACTION_PLOT_PAIRS
    }
    return pair in selected


interaction_summary_rows = []
interaction_surface_files = []

if RUN_2D_INTERACTIONS:
    interaction_anchors = anchors_df.head(
        min(INTERACTION_MAX_ANCHORS, len(anchors_df))
    )

    for _, anchor_row in interaction_anchors.iterrows():
        anchor_id = int(anchor_row["anchor_id"])
        if INTERACTION_USE_ACTIVE_CORE:
            theta_anchor = active_core_theta_by_anchor[
                anchor_id
            ].copy()
            interaction_reference = "active_core"
        else:
            theta_anchor = np.asarray(
                anchor_row["theta_vector"],
                dtype=float,
            ).copy()
            interaction_reference = "original_anchor"
        f0 = float(evaluate_theta(theta_anchor)["p_optimal"])

        for theta_i, theta_j in INTERACTION_PAIRS:
            theta_i, theta_j = int(theta_i), int(theta_j)
            period_i = parameter_period(theta_i)
            period_j = parameter_period(theta_j)
            grid_i = inclusive_grid(
                0.0,
                period_i,
                n_points=INTERACTION_GRID_POINTS,
            )
            grid_j = inclusive_grid(
                0.0,
                period_j,
                n_points=INTERACTION_GRID_POINTS,
            )

            surface_path = CHECKPOINT_DIR / (
                f"v204_interaction_{interaction_reference}_anchor_{anchor_id:02d}_"
                f"theta_{theta_i:02d}_{theta_j:02d}.npz"
            )

            if surface_path.exists():
                data = np.load(surface_path)
                p_surface = np.asarray(
                    data["p_surface"], dtype=float
                )
                f_i = np.asarray(data["f_i"], dtype=float)
                f_j = np.asarray(data["f_j"], dtype=float)
            else:
                f_i = np.empty(len(grid_i), dtype=float)
                for index_i, value_i in enumerate(grid_i):
                    theta_test = theta_anchor.copy()
                    theta_test[theta_i] = value_i
                    f_i[index_i] = evaluate_theta(
                        theta_test
                    )["p_optimal"]

                f_j = np.empty(len(grid_j), dtype=float)
                for index_j, value_j in enumerate(grid_j):
                    theta_test = theta_anchor.copy()
                    theta_test[theta_j] = value_j
                    f_j[index_j] = evaluate_theta(
                        theta_test
                    )["p_optimal"]

                p_surface = np.empty(
                    (len(grid_i), len(grid_j)),
                    dtype=float,
                )
                for index_i, value_i in enumerate(grid_i):
                    for index_j, value_j in enumerate(grid_j):
                        theta_test = theta_anchor.copy()
                        theta_test[theta_i] = value_i
                        theta_test[theta_j] = value_j
                        p_surface[index_i, index_j] = (
                            evaluate_theta(theta_test)["p_optimal"]
                        )

                # Mantém float64; converter para float32 esconderia o nível
                # real do resíduo em execuções retomadas.
                np.savez_compressed(
                    surface_path,
                    p_surface=p_surface,
                    f_i=f_i,
                    f_j=f_j,
                    grid_i=grid_i,
                    grid_j=grid_j,
                    theta_i=theta_i,
                    theta_j=theta_j,
                    f0=f0,
                )

            additive_prediction = (
                f_i[:, None] + f_j[None, :] - f0
            )
            multiplicative_prediction = (
                f_i[:, None]
                * f_j[None, :]
                / max(f0, SEPARABILITY_EPSILON)
            )

            additive_metrics, additive_residual = (
                regression_surface_metrics(
                    p_surface,
                    additive_prediction,
                )
            )
            multiplicative_metrics, multiplicative_residual = (
                regression_surface_metrics(
                    p_surface,
                    multiplicative_prediction,
                )
            )
            log_metrics = reliable_log_surface_metrics(
                p_surface,
                multiplicative_prediction,
            )

            (
                rank1_surface,
                singular_values,
                rank1_explained,
                rank1_relative_error,
                effective_rank,
            ) = rank1_svd_diagnostics(p_surface)

            multiplicative_better = bool(
                multiplicative_metrics[
                    "relative_fro_error"
                ]
                < additive_metrics["relative_fro_error"]
            )
            approximately_separable = bool(
                multiplicative_metrics[
                    "relative_fro_error"
                ]
                <= MULTIPLICATIVE_RELATIVE_ERROR_THRESHOLD
                and rank1_explained
                >= SVD_RANK1_EXPLAINED_THRESHOLD
            )

            interaction_summary_rows.append({
                "anchor_id": anchor_id,
                "theta_i": theta_i,
                "theta_j": theta_j,
                "f0": f0,
                "interaction_reference": interaction_reference,
                "p_surface_min": float(p_surface.min()),
                "p_surface_max": float(p_surface.max()),
                "additive_rmse": additive_metrics["rmse"],
                "additive_max_abs": additive_metrics["max_abs"],
                "additive_relative_fro_error": (
                    additive_metrics["relative_fro_error"]
                ),
                "additive_r2": additive_metrics["r2"],
                "multiplicative_rmse": (
                    multiplicative_metrics["rmse"]
                ),
                "multiplicative_max_abs": (
                    multiplicative_metrics["max_abs"]
                ),
                "multiplicative_relative_fro_error": (
                    multiplicative_metrics[
                        "relative_fro_error"
                    ]
                ),
                "multiplicative_r2": (
                    multiplicative_metrics["r2"]
                ),
                **log_metrics,
                "svd_rank1_explained_fraction": (
                    rank1_explained
                ),
                "svd_rank1_relative_error": (
                    rank1_relative_error
                ),
                "svd_effective_rank": effective_rank,
                "multiplicative_better_than_additive": (
                    multiplicative_better
                ),
                "approximately_multiplicative_separable": (
                    approximately_separable
                ),
                "surface_path": str(surface_path.resolve()),
            })
            interaction_surface_files.append(surface_path)

            # Somente pares selecionados recebem figuras detalhadas.
            if pair_is_selected_for_plot(theta_i, theta_j):
                plot_specs = [
                    (
                        p_surface,
                        "P(x*) observado",
                        "P(x*)",
                        "observed",
                        False,
                    ),
                    (
                        multiplicative_prediction,
                        "Predição multiplicativa: P_i P_j / P_0",
                        "P multiplicativa",
                        "multiplicative_prediction",
                        False,
                    ),
                    (
                        multiplicative_residual,
                        "Resíduo do modelo multiplicativo",
                        "observado - multiplicativo",
                        "multiplicative_residual",
                        True,
                    ),
                    (
                        rank1_surface,
                        "Melhor aproximação de posto 1 por SVD",
                        "P posto 1",
                        "svd_rank1",
                        False,
                    ),
                ]
                for (
                    matrix,
                    title,
                    color_label,
                    stem,
                    symmetric,
                ) in plot_specs:
                    fig, ax = plt.subplots(figsize=(8, 6))
                    kwargs = {}
                    if symmetric:
                        limit = max(
                            float(np.max(np.abs(matrix))),
                            1e-18,
                        )
                        kwargs = {"vmin": -limit, "vmax": limit}
                    image = ax.imshow(
                        matrix.T,
                        origin="lower",
                        aspect="auto",
                        extent=[
                            grid_i[0],
                            grid_i[-1],
                            grid_j[0],
                            grid_j[-1],
                        ],
                        **kwargs,
                    )
                    ax.set_xlabel(f"theta_{theta_i}")
                    ax.set_ylabel(f"theta_{theta_j}")
                    ax.set_title(
                        f"{title} — theta_{theta_i} × "
                        f"theta_{theta_j}, âncora {anchor_id}"
                    )
                    fig.colorbar(
                        image, ax=ax, label=color_label
                    )
                    fig.tight_layout()
                    fig.savefig(
                        FIGURE_DIR
                        / (
                            f"{stem}_anchor_{anchor_id:02d}_"
                            f"theta_{theta_i:02d}_{theta_j:02d}.png"
                        ),
                        dpi=180,
                    )
                    plt.show()

                fig, ax = plt.subplots(figsize=(8, 4))
                normalized_singular_values = (
                    singular_values
                    / max(
                        singular_values[0],
                        SEPARABILITY_EPSILON,
                    )
                )
                ax.semilogy(
                    np.arange(
                        1, len(singular_values) + 1
                    ),
                    np.maximum(
                        normalized_singular_values,
                        1e-18,
                    ),
                    marker="o",
                )
                ax.set_xlabel("índice do valor singular")
                ax.set_ylabel("valor singular / sigma_1")
                ax.set_title(
                    f"Espectro singular — theta_{theta_i} × "
                    f"theta_{theta_j}, âncora {anchor_id}"
                )
                fig.tight_layout()
                fig.savefig(
                    FIGURE_DIR
                    / (
                        f"svd_spectrum_anchor_{anchor_id:02d}_"
                        f"theta_{theta_i:02d}_{theta_j:02d}.png"
                    ),
                    dpi=180,
                )
                plt.show()

        print(f"21 pares concluídos para âncora {anchor_id}")

interaction_summary_df = pd.DataFrame(
    interaction_summary_rows
)
interaction_systematic_summary_df = pd.DataFrame()

if not interaction_summary_df.empty:
    interaction_systematic_summary_df = (
        interaction_summary_df.groupby(
            ["theta_i", "theta_j"],
            as_index=False,
        ).agg(
            n_anchors=("anchor_id", "nunique"),
            additive_relative_error_median=(
                "additive_relative_fro_error",
                "median",
            ),
            multiplicative_relative_error_median=(
                "multiplicative_relative_fro_error",
                "median",
            ),
            multiplicative_log_rmse_median=(
                "log_rmse_reliable",
                "median",
            ),
            weighted_log_rmse_median=(
                "weighted_log_rmse",
                "median",
            ),
            log_coverage_median=(
                "log_coverage",
                "median",
            ),
            rank1_explained_median=(
                "svd_rank1_explained_fraction",
                "median",
            ),
            rank1_relative_error_median=(
                "svd_rank1_relative_error",
                "median",
            ),
            effective_rank_median=(
                "svd_effective_rank",
                "median",
            ),
            fraction_multiplicative_better=(
                "multiplicative_better_than_additive",
                "mean",
            ),
            fraction_approximately_separable=(
                "approximately_multiplicative_separable",
                "mean",
            ),
        )
    )

display(interaction_summary_df)
display(interaction_systematic_summary_df)

# Matrizes-resumo dos 21 pares: uma leitura compacta da campanha completa.
if not interaction_systematic_summary_df.empty:
    active_order = list(ACTIVE_SET)
    error_matrix = pd.DataFrame(
        np.nan,
        index=active_order,
        columns=active_order,
    )
    rank1_matrix = error_matrix.copy()

    for _, row in interaction_systematic_summary_df.iterrows():
        i, j = int(row["theta_i"]), int(row["theta_j"])
        error = float(
            row["multiplicative_relative_error_median"]
        )
        explained = float(
            row["rank1_explained_median"]
        )
        error_matrix.loc[i, j] = error
        error_matrix.loc[j, i] = error
        rank1_matrix.loc[i, j] = explained
        rank1_matrix.loc[j, i] = explained

    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(
        np.log10(
            np.clip(
                error_matrix.to_numpy(dtype=float),
                1e-18,
                None,
            )
        ),
        origin="lower",
        aspect="auto",
    )
    ax.set_xticks(range(len(active_order)))
    ax.set_yticks(range(len(active_order)))
    ax.set_xticklabels(active_order)
    ax.set_yticklabels(active_order)
    ax.set_xlabel("theta_j")
    ax.set_ylabel("theta_i")
    ax.set_title(
        "Log10 do erro multiplicativo mediano — 21 pares"
    )
    fig.colorbar(
        image,
        ax=ax,
        label="log10(erro relativo)",
    )
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR
        / "all_active_pairs_multiplicative_error_matrix.png",
        dpi=180,
    )
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(
        rank1_matrix.to_numpy(dtype=float),
        origin="lower",
        aspect="auto",
        vmin=0.0,
        vmax=1.0,
    )
    ax.set_xticks(range(len(active_order)))
    ax.set_yticks(range(len(active_order)))
    ax.set_xticklabels(active_order)
    ax.set_yticklabels(active_order)
    ax.set_xlabel("theta_j")
    ax.set_ylabel("theta_i")
    ax.set_title(
        "Fração explicada pelo posto 1 — 21 pares"
    )
    fig.colorbar(
        image,
        ax=ax,
        label="fração explicada",
    )
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR
        / "all_active_pairs_rank1_explained_matrix.png",
        dpi=180,
    )
    plt.show()

    fraction_all = float(
        interaction_systematic_summary_df[
            "fraction_approximately_separable"
        ].mean()
    )
    worst_error = float(
        interaction_systematic_summary_df[
            "multiplicative_relative_error_median"
        ].max()
    )
    display(Markdown(
        "**Diagnóstico automático dos 21 pares:** "
        f"fração média classificada como separável = "
        f"**{fraction_all:.3f}**; pior erro multiplicativo "
        f"mediano = **{worst_error:.3e}**."
    ))


#### Leitura dos gráficos de separabilidade

Nos pares já executados \((17,2)\) e \((17,14)\), o modelo multiplicativo reproduziu a superfície até a precisão numérica, enquanto o modelo aditivo falhou. O espectro singular apresentou um único valor dominante.

A campanha atual amplia a conclusão para os 21 pares. Os dois mapas-resumo devem responder:

- todos os pares permanecem de posto 1?
- existe algum par com erro multiplicativo claramente acima do piso numérico?
- a separabilidade é sistemática entre as âncoras?

Uma não aditividade grande não será interpretada como interação inseparável quando o produto das respostas individuais explicar a superfície.


# Parte XI — fatorização simultânea de 2 até 7 parâmetros ativos

Separabilidade par a par não garante automaticamente separabilidade em dimensões maiores. Por isso, esta parte sorteia:

- um subconjunto de tamanho \(m=2,3,\ldots,7\);
- um ângulo aleatório para cada parâmetro do subconjunto.

A probabilidade avaliada diretamente é comparada com:

\[
\widehat P_S(x^*)
=
P_0
\prod_{j\in S}
\frac{P_j(\theta_j)}{P_0}
=
\frac{\prod_{j\in S}P_j(\theta_j)}
{P_0^{|S|-1}}.
\]

O cálculo é feito em escala logarítmica para evitar underflow quando a probabilidade é muito pequena.


### Célula 26 — Teste multidimensional do modelo multiplicativo

Cada ensaio utiliza o núcleo ativo de uma âncora como base. Os parâmetros não selecionados permanecem na base, enquanto o subconjunto ativo recebe ângulos aleatórios.

O teste registra:

- probabilidade observada;
- probabilidade prevista pelo produto das respostas individuais;
- erro absoluto;
- erro relativo simétrico;
- erro em \(\log_{10}P\);
- tamanho e composição do subconjunto.


In [ ]:
# ============================================================
# 26. FATORIZAÇÃO SIMULTÂNEA DE 2, 3, ..., 7 ATIVOS
# ============================================================


def run_active_factor_trials(anchor_row):
    """Testa a fatorização multiplicativa em subconjuntos ativos aleatórios."""
    anchor_id = int(anchor_row["anchor_id"])
    path = CHECKPOINT_DIR / (
        f"v204_active_factor_trials_anchor_{anchor_id:02d}.pkl"
    )
    if path.exists():
        return pd.read_pickle(path)

    rng = np.random.default_rng(
        RANDOM_SEED + 18_000_000 + 10_000 * anchor_id
    )
    theta_core = active_core_theta_by_anchor[anchor_id].copy()
    p0 = float(evaluate_theta(theta_core)["p_optimal"])
    log_p0 = float(
        np.log(max(p0, ACTIVE_FACTOR_NUMERIC_FLOOR))
    )

    rows = []
    for subset_size in range(2, len(ACTIVE_SET) + 1):
        for trial_index in range(
            ACTIVE_FACTOR_TRIALS_PER_SIZE
        ):
            subset = tuple(sorted(
                int(value)
                for value in rng.choice(
                    ACTIVE_SET,
                    size=subset_size,
                    replace=False,
                )
            ))
            targets = {
                theta_index: float(
                    rng.uniform(
                        0.0,
                        parameter_period(theta_index),
                    )
                )
                for theta_index in subset
            }

            # Avaliação conjunta observada.
            theta_joint = theta_core.copy()
            for theta_index, target in targets.items():
                theta_joint[theta_index] = target
            p_observed = float(
                evaluate_theta(theta_joint)["p_optimal"]
            )

            # Produto das respostas one-at-a-time calculadas na mesma base.
            log_prediction = log_p0
            single_probabilities = {}
            for theta_index, target in targets.items():
                theta_single = theta_core.copy()
                theta_single[theta_index] = target
                p_single = float(
                    evaluate_theta(theta_single)["p_optimal"]
                )
                single_probabilities[theta_index] = p_single
                log_prediction += (
                    np.log(
                        max(
                            p_single,
                            ACTIVE_FACTOR_NUMERIC_FLOOR,
                        )
                    )
                    - log_p0
                )

            # exp(-745) é aproximadamente o menor valor double representável.
            p_predicted = float(
                np.exp(np.clip(log_prediction, -745.0, 700.0))
            )
            absolute_error = float(
                abs(p_observed - p_predicted)
            )
            symmetric_relative_error = float(
                2.0
                * absolute_error
                / max(
                    abs(p_observed) + abs(p_predicted),
                    ACTIVE_FACTOR_NUMERIC_FLOOR,
                )
            )

            if (
                p_observed > LOG_PROBABILITY_THRESHOLD
                and p_predicted > LOG_PROBABILITY_THRESHOLD
            ):
                log10_error = float(
                    np.log10(p_observed)
                    - np.log10(p_predicted)
                )
                reliable_log_point = True
            else:
                log10_error = np.nan
                reliable_log_point = False

            rows.append({
                "anchor_id": anchor_id,
                "subset_size": int(subset_size),
                "trial_index": int(trial_index),
                "subset": subset,
                "targets": targets,
                "p0": p0,
                "p_observed": p_observed,
                "p_predicted": p_predicted,
                "absolute_error": absolute_error,
                "symmetric_relative_error": (
                    symmetric_relative_error
                ),
                "log10_error": log10_error,
                "reliable_log_point": reliable_log_point,
                "single_probabilities": (
                    single_probabilities
                ),
            })

    frame = pd.DataFrame(rows)
    frame.to_pickle(path)
    return frame


def summarize_factor_group(group):
    """Resume um grupo sem usar erro relativo instável perto de zero."""
    observed = group["p_observed"].to_numpy(dtype=float)
    predicted = group["p_predicted"].to_numpy(dtype=float)
    residual = observed - predicted
    centered = observed - observed.mean()
    denominator = float(np.sum(centered ** 2))
    r2 = (
        float(
            1.0
            - np.sum(residual ** 2)
            / denominator
        )
        if denominator > 1e-30
        else np.nan
    )
    reliable_log = group["log10_error"].dropna()
    return pd.Series({
        "n_trials": int(len(group)),
        "rmse": float(
            np.sqrt(np.mean(residual ** 2))
        ),
        "max_abs_error": float(
            np.max(np.abs(residual))
        ),
        "median_symmetric_relative_error": float(
            group["symmetric_relative_error"].median()
        ),
        "q95_symmetric_relative_error": float(
            group["symmetric_relative_error"].quantile(0.95)
        ),
        "linear_r2": r2,
        "reliable_log_fraction": float(
            group["reliable_log_point"].mean()
        ),
        "log10_rmse_reliable": (
            float(
                np.sqrt(
                    np.mean(reliable_log.to_numpy() ** 2)
                )
            )
            if len(reliable_log)
            else np.nan
        ),
    })


active_factor_trials_df = pd.DataFrame()
active_factor_summary_df = pd.DataFrame()

if RUN_ACTIVE_FACTOR_MODEL:
    factor_anchors = anchors_df.head(
        min(ACTIVE_FACTOR_MAX_ANCHORS, len(anchors_df))
    )
    frames = []
    for _, anchor_row in factor_anchors.iterrows():
        frames.append(
            run_active_factor_trials(anchor_row)
        )
        print(
            f"fatorização multidimensional concluída para "
            f"âncora {int(anchor_row['anchor_id'])}"
        )
    active_factor_trials_df = pd.concat(
        frames, ignore_index=True
    )

    summary_frames = []
    for subset_size, group in active_factor_trials_df.groupby(
        "subset_size"
    ):
        row = summarize_factor_group(group).to_dict()
        row["subset_size"] = int(subset_size)
        summary_frames.append(row)
    active_factor_summary_df = pd.DataFrame(
        summary_frames
    ).sort_values("subset_size").reset_index(drop=True)
    display(active_factor_summary_df)

    # Erro versus dimensão do subconjunto.
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(
        active_factor_summary_df["subset_size"],
        np.maximum(
            active_factor_summary_df["rmse"],
            1e-18,
        ),
        marker="o",
        label="RMSE",
    )
    ax.plot(
        active_factor_summary_df["subset_size"],
        np.maximum(
            active_factor_summary_df[
                "max_abs_error"
            ],
            1e-18,
        ),
        marker="s",
        label="erro máximo",
    )
    ax.set_yscale("log")
    ax.set_xlabel(
        "quantidade de parâmetros ativos modificados"
    )
    ax.set_ylabel("erro da fatorização")
    ax.set_title(
        "A separabilidade permanece de 2 até 7 parâmetros?"
    )
    ax.set_xticks(range(2, len(ACTIVE_SET) + 1))
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR
        / "active_factorization_error_by_subset_size.png",
        dpi=180,
    )
    plt.show()

    # Comparação direta em escala logarítmica.
    positive = active_factor_trials_df.loc[
        active_factor_trials_df["p_observed"].gt(0.0)
        & active_factor_trials_df["p_predicted"].gt(0.0)
    ].copy()
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(
        np.log10(
            np.maximum(
                positive["p_predicted"],
                ACTIVE_FACTOR_NUMERIC_FLOOR,
            )
        ),
        np.log10(
            np.maximum(
                positive["p_observed"],
                ACTIVE_FACTOR_NUMERIC_FLOOR,
            )
        ),
        s=12,
        alpha=0.35,
    )
    lower = float(min(
        ax.get_xlim()[0],
        ax.get_ylim()[0],
    ))
    upper = float(max(
        ax.get_xlim()[1],
        ax.get_ylim()[1],
    ))
    ax.plot(
        [lower, upper],
        [lower, upper],
        linestyle="--",
    )
    ax.set_xlim(lower, upper)
    ax.set_ylim(lower, upper)
    ax.set_xlabel("log10 P previsto")
    ax.set_ylabel("log10 P observado")
    ax.set_title(
        "Fatorização multiplicativa do núcleo ativo"
    )
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR
        / "active_factorization_observed_vs_predicted.png",
        dpi=180,
    )
    plt.show()

    seven_row = active_factor_summary_df.loc[
        active_factor_summary_df[
            "subset_size"
        ].eq(len(ACTIVE_SET))
    ]
    if not seven_row.empty:
        seven_rmse = float(seven_row.iloc[0]["rmse"])
        seven_max = float(
            seven_row.iloc[0]["max_abs_error"]
        )
        display(Markdown(
            "**Diagnóstico automático para os sete ativos:** "
            f"RMSE = **{seven_rmse:.3e}**; "
            f"erro máximo = **{seven_max:.3e}**."
        ))


#### Como interpretar o teste dos sete ativos

- erro próximo da precisão numérica até \(m=7\): a resposta de \(P(x^*)\) é bem descrita por fatores unidimensionais;
- erro pequeno para pares, mas crescente com \(m\): existem interações de ordem superior;
- erro concentrado em subconjuntos específicos: esses grupos devem receber termos residuais explícitos no modelo futuro.

Mesmo uma fatorização perfeita de \(P(x^*)\) não prova que o statevector ou todas as probabilidades dos 210 portfólios fatorizem da mesma forma.


# Parte XII — tabelas para a próxima fase do Transformer

O notebook não treina o Transformer. Ele produz entidades auditáveis que evitam tratar o índice do parâmetro como significado físico:

1. descrição do Hamiltoniano e dos termos `Z/ZZ`;
2. mapa de portas, qubits, ativos e períodos de cada parâmetro;
3. varreduras individuais e atlas dos 30 parâmetros;
4. núcleo ativo e testes condicionais dos 23 parâmetros restantes;
5. fidelidade do statevector após aleatorizar os inativos;
6. espectro, rank e diagonal da QFIM;
7. separabilidade multiplicativa dos 21 pares ativos;
8. fatorização simultânea de 2 até 7 parâmetros ativos.

O futuro token de parâmetro deverá usar essas relações estruturais e geométricas, e não `theta_index=17` como rótulo universal.


### Célula 27 — Exportação dos resultados e manifesto

Esta célula salva tabelas, checkpoints consolidados, resumos e o manifesto completo da execução.

O manifesto registra:

- configuração e sementes;
- quantidade de avaliações;
- conjunto ativo e conjunto inativo;
- número de superfícies e ensaios de fatorização;
- rank mediano da QFIM;
- ausência de COBYLA;
- caminhos das saídas.

Objetos complexos são convertidos para JSON textual antes da exportação para CSV.


In [ ]:
# ============================================================
# 27. EXPORTAÇÃO FINAL E MANIFESTO
# ============================================================


def csv_safe(frame):
    """Converte objetos complexos em JSON textual antes de escrever CSV."""
    output = frame.copy()
    for column in output.columns:
        if output.empty:
            break
        contains_complex = output[column].map(
            lambda value: isinstance(value, (list, tuple, dict, np.ndarray))
        ).any()
        if contains_complex:
            output[column] = output[column].map(
                lambda value: json.dumps(
                    value.tolist() if isinstance(value, np.ndarray) else value,
                    ensure_ascii=False,
                ) if isinstance(value, (list, tuple, dict, np.ndarray)) else value
            )
    return output


# Grupo 1: descrição clássica, Hamiltoniano e mapa estrutural dos parâmetros.
csv_safe(problem_summary_df).to_csv(TABLE_DIR / "asset_summary.csv", index=False)
csv_safe(enumeration_df).to_csv(TABLE_DIR / "classical_enumeration_k4.csv", index=False)
csv_safe(asset_decision_df).to_csv(TABLE_DIR / "asset_decision_margins.csv", index=False)
csv_safe(pair_gap_df).to_csv(TABLE_DIR / "pair_conditional_gaps.csv", index=False)
csv_safe(hamiltonian_terms_df).to_csv(TABLE_DIR / "hamiltonian_terms.csv", index=False)
csv_safe(parameter_map_df).to_csv(TABLE_DIR / "parameter_structure_and_financial_links.csv", index=False)
csv_safe(parameter_occurrence_df).to_csv(TABLE_DIR / "parameter_primitive_occurrences.csv", index=False)

# Grupo 2: auditoria de compatibilidade e vetores âncora selecionados.
csv_safe(compatibility_audit_df).to_csv(TABLE_DIR / "merge_circuit_compatibility_audit.csv", index=False)
anchors_df.to_pickle(TABLE_DIR / "selected_anchors.pkl")
csv_safe(anchors_df.drop(columns=["theta_vector"], errors="ignore")).to_csv(
    TABLE_DIR / "selected_anchors_summary.csv", index=False
)

# Grupo 3: varreduras individuais, atlas e perturbações dirigidas.
individual_sweep_df.to_pickle(TABLE_DIR / "individual_detailed_sweeps.pkl")
csv_safe(individual_sweep_df).to_csv(TABLE_DIR / "individual_detailed_sweeps.csv", index=False)
csv_safe(individual_anchor_summary_df).to_csv(TABLE_DIR / "individual_anchor_summary.csv", index=False)
csv_safe(individual_systematic_summary_df).to_csv(TABLE_DIR / "individual_systematic_summary.csv", index=False)
csv_safe(mean_shape_df).to_csv(
    TABLE_DIR / "individual_mean_shape_common_phase.csv",
    index=False,
)
csv_safe(mean_distribution_metadata_df).to_csv(TABLE_DIR / "mean_distribution_files.csv", index=False)

if RUN_ALL_THETA_ATLAS and not atlas_sweep_df.empty:
    atlas_sweep_df.to_pickle(TABLE_DIR / "all_theta_atlas_sweeps.pkl")
    csv_safe(atlas_parameter_summary_df).to_csv(TABLE_DIR / "all_theta_atlas_summary.csv", index=False)
    csv_safe(gate_relevance_correlations_df).to_csv(TABLE_DIR / "gate_relevance_correlations.csv", index=False)
    csv_safe(gate_type_summary_df).to_csv(TABLE_DIR / "gate_type_summary.csv", index=False)

cumulative_sweep_df.to_pickle(TABLE_DIR / "cumulative_sweeps.pkl")
csv_safe(cumulative_endpoint_df).to_csv(TABLE_DIR / "cumulative_endpoint_summary.csv", index=False)

# Grupo 4: suficiência e robustez do núcleo ativo.
csv_safe(active_core_baseline_df).to_csv(TABLE_DIR / "active_core_baseline.csv", index=False)
if not inactive_conditional_sweep_df.empty:
    inactive_conditional_sweep_df.to_pickle(TABLE_DIR / "inactive_conditional_sweeps.pkl")
    csv_safe(inactive_conditional_summary_df).to_csv(
        TABLE_DIR / "inactive_conditional_summary.csv", index=False
    )
if not inactive_nested_random_df.empty:
    inactive_nested_random_df.to_pickle(TABLE_DIR / "inactive_nested_random_paths.pkl")
    csv_safe(inactive_nested_summary_df).to_csv(
        TABLE_DIR / "inactive_nested_random_summary.csv", index=False
    )
if not active_core_rescue_df.empty:
    active_core_rescue_df.to_pickle(TABLE_DIR / "active_core_rescue_trials.pkl")
    csv_safe(active_core_rescue_summary_df).to_csv(
        TABLE_DIR / "active_core_rescue_summary.csv", index=False
    )

if not statevector_inactive_df.empty:
    statevector_inactive_df.to_pickle(
        TABLE_DIR / "inactive_statevector_trials.pkl"
    )
    csv_safe(statevector_inactive_summary_df).to_csv(
        TABLE_DIR / "inactive_statevector_summary.csv",
        index=False,
    )

if not qfim_anchor_summary_df.empty:
    csv_safe(qfim_anchor_summary_df).to_csv(
        TABLE_DIR / "qfim_anchor_summary.csv",
        index=False,
    )
    csv_safe(qfim_spectrum_df).to_csv(
        TABLE_DIR / "qfim_spectrum.csv",
        index=False,
    )
    csv_safe(qfim_diagonal_df).to_csv(
        TABLE_DIR / "qfim_parameter_diagonal_by_anchor.csv",
        index=False,
    )
    csv_safe(qfim_diagonal_summary_df).to_csv(
        TABLE_DIR / "qfim_parameter_diagonal_summary.csv",
        index=False,
    )

csv_safe(interaction_summary_df).to_csv(TABLE_DIR / "interaction_summary.csv", index=False)
csv_safe(interaction_systematic_summary_df).to_csv(
    TABLE_DIR / "interaction_systematic_summary.csv", index=False
)

if not active_factor_trials_df.empty:
    active_factor_trials_df.to_pickle(
        TABLE_DIR / "active_factorization_trials.pkl"
    )
    csv_safe(active_factor_summary_df).to_csv(
        TABLE_DIR / "active_factorization_summary.csv",
        index=False,
    )

# Manifesto: reúne configuração, contagens, hashes e garantias metodológicas.
manifest = {
    **CONFIG,
    "merge_path_resolved": str(merge_path),
    "output_root": str(OUTPUT_ROOT.resolve()),
    "problem_hash": DATA_HASH,
    "n_bank_rows": int(len(merge_df)),
    "n_valid_theta_rows": int(len(valid_bank)),
    "n_top_masked_rows": int(len(top_masked_bank)),
    "n_anchors": int(len(anchors_df)),
    "n_assets": int(N_ASSETS),
    "n_parameters": int(N_PARAMETERS),
    "exact_energy": exact_energy,
    "exact_qiskit_bitstrings": exact_qiskit_bitstrings,
    "initial_x_qubits": initial_x_qubits,
    "initial_basis_state_is_solution": False,
    "structural_audit": structural_audit,
    "optimizer_used": False,
    "cobyla_calls": 0,
    "n_detailed_evaluations": int(len(individual_sweep_df)),
    "n_atlas_evaluations": int(len(atlas_sweep_df)),
    "n_cumulative_evaluations": int(len(cumulative_sweep_df)),
    "active_theta_indices": list(ACTIVE_SET),
    "inactive_theta_indices": list(INACTIVE_SET),
    "n_inactive_conditional_evaluations": int(len(inactive_conditional_sweep_df)),
    "n_inactive_nested_evaluations": int(len(inactive_nested_random_df)),
    "n_active_core_rescue_trials": int(len(active_core_rescue_df)),
    "n_statevector_phase_trials": int(len(statevector_inactive_df)),
    "n_qfim_anchors": int(len(qfim_anchor_summary_df)),
    "qfim_rank_median": (
        float(qfim_anchor_summary_df["numerical_rank"].median())
        if not qfim_anchor_summary_df.empty
        else None
    ),
    "n_interaction_surfaces": int(len(interaction_summary_df)),
    "n_active_factor_trials": int(len(active_factor_trials_df)),
    "separability_models": [
        "additive",
        "multiplicative_slice_product",
        "svd_rank1",
        "active_subset_product",
    ],
}

(OUTPUT_ROOT / "run_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(manifest, indent=2, ensure_ascii=False))
print("Arquivos salvos em:", OUTPUT_ROOT.resolve())


# Ordem correta de leitura dos resultados

1. **`merge_circuit_compatibility_audit.csv`** — compatibilidade entre banco e circuito.
2. **`individual_systematic_summary.csv`** — forma, amplitude e posição dos picos nas varreduras detalhadas.
3. **`all_theta_atlas_summary.csv`** — separação entre parâmetros ativos e individualmente inativos.
4. **`active_core_baseline.csv`** — construção do núcleo ativo nas melhores regiões.
5. **`inactive_conditional_summary.csv`** — verifica cada inativo com os sete ativos fixos.
6. **`inactive_nested_random_summary.csv`** — modifica progressivamente de 1 até 23 inativos.
7. **`active_core_rescue_summary.csv`** — restaura somente os sete ativos sobre um fundo aleatório.
8. **`inactive_statevector_summary.csv`** — distingue redundância completa de alterações apenas de fase.
9. **`qfim_anchor_summary.csv`** e **`qfim_spectrum.csv`** — dimensão geométrica efetiva do estado.
10. **`interaction_systematic_summary.csv`** — separabilidade dos 21 pares ativos.
11. **`active_factorization_summary.csv`** — testa o produto de respostas para 2, 3, ..., 7 ativos.
12. **`cumulative_endpoint_summary.csv`** — compara trajetórias dirigidas de fortes, controles e gate-off.

### Critérios de conclusão

Uma superfície só deverá ser chamada de multiplicativamente separável quando o erro relativo for pequeno e a primeira componente SVD explicar a fração configurada.

O rank da QFIM deve ser interpretado com um limiar relativo e absoluto; autovalores no piso numérico não representam novas direções físicas.

Nenhuma conclusão de generalização para outros problemas deverá usar apenas os índices dos parâmetros. A generalização exigirá novos Hamiltonianos e descrições estruturais de portas, qubits, ativos e termos do problema.
